# ForecastMaster_Refresh R05.3 — hourly and report-request refresh

One maintained notebook. `REQUESTS` keeps the existing report-button controller; `HOURLY` receives `FM_METADATA_JSON` from the published metadata function.

The hourly pipeline runs this notebook only when `NeedsRefresh` is true. A first baseline requires one successful run. The same writer lease covers both modes. Hourly mode reads workbook bytes only when source metadata or SharePoint version changed, reuses current-file report identities, and commits metadata-success.json only after ingestion, reporting, model confirmation and current-file maintenance succeed. Source workbooks remain inputs.

Keep schedules paused during installation and qualification. A company-side successful hourly run and a second unchanged gate test are required before enabling its daily 08:00–17:00 schedule. The native on-demand button path is a separate pending step.


In [ ]:
# Pipeline parameters. Defaults preserve the existing request controller.
FM_RUN_MODE = "REQUESTS"
FM_METADATA_JSON = ""


In [ ]:
# The whole Demand tree is the permanent discovery root.
SOURCE_ROOT = "Files/Demand"
TABLE_PREFIX = "forecast_v18"
REPORT_PREFIX = "fm_review"
ALLOWED_PROFILES = ["Local", "Ext"]
# ALL finds matching xlsx/xlsm workbooks in any folder, with any version suffix.
# FILES checks only the exact relative paths below, for a quicker review refresh.
REFRESH_MODE = "ALL"
SELECTED_RELATIVE_PATHS = []
FORCE_REHASH = False
FORCE_REBUILD_REPORTING_FACTS = False
FORCE_MODEL_REFRESH = False
REFRESH_SEMANTIC_MODEL = True
WORKSPACE_ID = "f5cb5742-9a7c-4337-a4c0-952706dddf0b"
SEMANTIC_MODEL_ID = "e7af2967-7725-4108-a170-9e490d4e35bb"
MAX_BRIDGE_BYTES = 128 * 1024 * 1024
MAX_OBSERVATIONS_PER_FILE = 5_000_000
SHUFFLE_PARTITIONS = 8
MODEL_REFRESH_TIMEOUT_SECONDS = 180

LAKEHOUSE_ID = "d6e33ebd-67b1-420a-8d84-c3a0d04d8b21"


In [ ]:
"""Consume immutable SharePoint request files with OneLake receipts and one writer.

The exclusive infinite Blob lease is a run-coordination lock. A killed session
needs deliberate lease recovery after checking that no writer is still running.
No request file or source workbook is moved, edited, or deleted.
"""

from __future__ import annotations

import hashlib
import json
import re
from collections.abc import Callable
from datetime import UTC, datetime
from email.utils import format_datetime
from typing import Any
from urllib.parse import quote
from uuid import UUID, uuid4

import requests

REQUEST_NAME = re.compile(
    r"refresh-([0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12})\.json\Z"
)


def queue_utc() -> str:
    return datetime.now(UTC).isoformat()


class WorkerRefreshError(RuntimeError):
    """The request has a retained failure receipt; it is never a green data refresh."""


class OneLakeQueueStore:
    def __init__(
        self,
        workspace: str,
        lakehouse: str,
        get_token: Callable[[str], str],
        *,
        http: Any = requests,
    ):
        workspace, lakehouse = str(UUID(workspace)), str(UUID(lakehouse))
        self.base = f"https://onelake.blob.fabric.microsoft.com/{workspace}/{lakehouse}/Files/ForecastMasterRefreshState/"
        self.get_token, self.http = get_token, http
        self.lease_id = ""

    def _headers(self) -> dict[str, str]:
        return {
            "Authorization": "Bearer " + self.get_token("storage"),
            "x-ms-version": "2021-06-08",
            "x-ms-date": format_datetime(datetime.now(UTC), usegmt=True),
        }

    def _url(self, name: str) -> str:
        if (
            not re.fullmatch(r"[a-zA-Z0-9_.-]+(?:/[a-zA-Z0-9_.-]+)*", name)
            or ".." in name
        ):
            raise ValueError("Invalid worker-state filename")
        return self.base + quote(name, safe="/")

    def read(self, name: str) -> dict[str, Any] | None:
        result = self.http.get(
            self._url(name), headers=self._headers(), timeout=45, allow_redirects=False
        )
        if result.status_code == 404:
            return None
        if result.status_code != 200:
            raise RuntimeError(
                f"Cannot read worker state {name} (HTTP {result.status_code})."
            )
        content = result.json()
        if not isinstance(content, dict):
            raise TypeError("Invalid worker state " + name)
        return content

    def write(self, name: str, content: dict[str, Any]) -> None:
        result = self.http.put(
            self._url(name),
            headers={
                **self._headers(),
                "x-ms-blob-type": "BlockBlob",
                "Content-Type": "application/json",
            },
            data=json.dumps(
                content, ensure_ascii=False, allow_nan=False, indent=2
            ).encode("utf-8"),
            timeout=45,
            allow_redirects=False,
        )
        if result.status_code != 201:
            raise RuntimeError(
                f"Cannot retain worker receipt {name} (HTTP {result.status_code})."
            )

    def acquire(self) -> bool:
        result = self.http.put(
            self._url("worker.lock"),
            headers={
                **self._headers(),
                "x-ms-blob-type": "BlockBlob",
                "If-None-Match": "*",
            },
            data=b"",
            timeout=45,
            allow_redirects=False,
        )
        if result.status_code not in {201, 409, 412}:
            raise RuntimeError(
                f"Cannot initialize writer lock (HTTP {result.status_code})."
            )
        proposed = str(uuid4())
        leased = self.http.put(
            self._url("worker.lock"),
            params={"comp": "lease"},
            headers={
                **self._headers(),
                "x-ms-lease-action": "acquire",
                "x-ms-lease-duration": "-1",
                "x-ms-proposed-lease-id": proposed,
            },
            timeout=45,
            allow_redirects=False,
        )
        if (
            leased.status_code == 409
            and leased.headers.get("x-ms-error-code") == "LeaseAlreadyPresent"
        ):
            return False
        if leased.status_code != 201 or leased.headers.get("x-ms-lease-id") != proposed:
            raise RuntimeError(
                f"Exclusive writer lock could not be verified (HTTP {leased.status_code}). No refresh started."
            )
        self.lease_id = proposed
        return True

    def release(self) -> None:
        if not self.lease_id:
            return
        result = self.http.put(
            self._url("worker.lock"),
            params={"comp": "lease"},
            headers={
                **self._headers(),
                "x-ms-lease-action": "release",
                "x-ms-lease-id": self.lease_id,
            },
            timeout=45,
            allow_redirects=False,
        )
        if result.status_code != 200:
            raise RuntimeError(
                f"Writer lock release failed (HTTP {result.status_code}); check the worker job before recovery."
            )
        self.lease_id = ""


class RequestWorker:
    def __init__(self, fs: Any, store: Any, request_folder: str, state_folder: str):
        self.fs, self.store = fs, store
        self.request_folder, self.state_folder = (
            request_folder.rstrip("/"),
            state_folder.rstrip("/"),
        )

    def _ledger(self) -> dict[str, Any]:
        ledger = self.store.read("requests.json")
        if ledger is None:
            return {"SchemaVersion": 1, "Requests": {}}
        if not isinstance(ledger, dict):
            raise TypeError("Worker request index must be an object.")
        if ledger.get("SchemaVersion") != 1 or not isinstance(
            ledger.get("Requests"), dict
        ):
            raise ValueError(
                "Worker request index is invalid. Retained batch receipts remain available."
            )
        if any(
            not isinstance(v, dict)
            or v.get("Status") not in {"SUCCESS", "FAILED", "REJECTED"}
            for v in ledger["Requests"].values()
        ):
            raise ValueError("Worker request index contains an invalid result.")
        return ledger

    def pending(self) -> list[str]:
        seen = self._ledger()["Requests"]
        names = []
        for item in self.fs.ls(self.request_folder):
            match = REQUEST_NAME.fullmatch(item.name)
            if item.isFile and match and str(UUID(match[1])) not in seen:
                names.append(item.name)
        return sorted(names)

    def run(
        self, refresh: Callable[[], tuple[Any, dict[str, Any]]]
    ) -> tuple[Any, dict[str, Any]]:
        if not self.pending():
            return None, {
                "Status": "IDLE",
                "Message": "No new refresh requests. No source discovery or model refresh ran.",
            }
        if not self.fs.mkdirs(self.state_folder) or not self.fs.mkdirs(
            self.state_folder + "/batches"
        ):
            raise RuntimeError("Cannot prepare the native lakehouse receipt folders.")
        if not self.store.acquire():
            return None, {
                "Status": "BUSY",
                "Message": "Another worker owns the writer lock. Requests remain queued.",
            }
        try:
            ledger = self._ledger()
            names = self.pending()
            if not names:
                return None, {
                    "Status": "IDLE",
                    "Message": "Another worker already handled these requests.",
                }
            batch_id, started = str(uuid4()), queue_utc()
            accepted: list[str] = []
            records: dict[str, Any] = {}
            for name in names:
                match = REQUEST_NAME.fullmatch(name)
                if match is None:
                    raise ValueError("Request filename changed during discovery.")
                identity = str(UUID(match[1]))
                text = self.fs.head(self.request_folder + "/" + name, 65537)
                record: dict[str, Any] = {
                    "RequestFile": name,
                    "ContentSHA256": hashlib.sha256(text.encode()).hexdigest(),
                    "BatchID": batch_id,
                }
                try:
                    if len(text.encode()) > 65536:
                        raise ValueError("Request exceeds the 64 KiB size limit.")
                    body = json.loads(text)
                    if (
                        not isinstance(body, dict)
                        or set(body) != {"SchemaVersion", "Action", "Scope"}
                        or type(body.get("SchemaVersion")) is not int
                        or body
                        != {
                            "SchemaVersion": 1,
                            "Action": "REFRESH_DEMAND",
                            "Scope": "ALL",
                        }
                    ):
                        raise ValueError(
                            "Expected SchemaVersion 1, Action REFRESH_DEMAND, Scope ALL."
                        )
                    accepted.append(identity)
                    record["Status"] = "QUEUED"
                except (ValueError, TypeError) as exc:
                    record.update(Status="REJECTED", Error=str(exc))
                records[identity] = record
            engine: Any = None
            summary: dict[str, Any] = {"Status": "NO_VALID_REQUESTS"}
            if accepted:
                try:
                    engine, summary = refresh()
                except Exception as exc:  # noqa: BLE001 -- preserve outcome, then fail the scheduled job
                    summary = {"Status": "FAILED", "StageErrors": [str(exc)]}
                success = (
                    summary.get("Status") == "SUCCESS"
                    and not summary.get("StageErrors")
                    and summary.get("ModelRefresh", {}).get("Status")
                    in {"COMPLETED", "UNCHANGED"}
                    and not summary.get("FileResults", {}).get("FAILED", 0)
                )
                for identity in accepted:
                    records[identity]["Status"] = "SUCCESS" if success else "FAILED"
            ended = queue_utc()
            for record in records.values():
                record.update(StartedAtUtc=started, FinishedAtUtc=ended)
            result = {
                "SchemaVersion": 1,
                "BatchID": batch_id,
                "StartedAtUtc": started,
                "FinishedAtUtc": ended,
                "Status": "SUCCESS"
                if records and all(r["Status"] == "SUCCESS" for r in records.values())
                else "FAILED",
                "Requests": records,
                "Refresh": summary,
            }
            # A durable batch receipt precedes the compact processed-request index.
            self.store.write("batches/" + batch_id + ".json", result)
            ledger["Requests"].update(records)
            self.store.write("requests.json", ledger)
            self.store.write("latest.json", result)
            print(json.dumps(result, indent=2), flush=True)
            if result["Status"] != "SUCCESS":
                raise WorkerRefreshError(
                    "Refresh request failed or was rejected. Read Files/ForecastMasterRefreshState/latest.json; a new button click submits a retry."
                )
            return engine, result
        finally:
            self.store.release()


In [ ]:
"""Coordinate a metadata-gated hourly run through the existing refresh engine.

The Fabric pipeline supplies the complete JSON string from checkDemandChangesR05.
The existing request-only worker and the hourly worker share worker.lock.
Baselines contain only file metadata; workbooks remain inputs.
"""

from __future__ import annotations

import hashlib
import json
import re
from collections.abc import Callable
from datetime import UTC, datetime
from typing import Any
from uuid import uuid4


MONTH = r"(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec) \d{2}"
BRIDGE = re.compile(
    r"[^_]+_.+?_Bridge_"
    + MONTH
    + r"_EF-"
    + MONTH
    + r"_IMS-"
    + MONTH
    + r"_.+\.(?:xlsx|xlsm)",
    re.IGNORECASE,
)


def metadata_inventory(rows: Any) -> dict[str, dict[str, Any]]:
    if not isinstance(rows, list) or not rows:
        raise ValueError("A complete nonempty bridge inventory is required.")
    result: dict[str, dict[str, Any]] = {}
    for row in rows:
        if not isinstance(row, dict) or set(row) != {
            "Path",
            "Bytes",
            "ModifiedAtUtc",
            "ETag",
        }:
            raise ValueError("Invalid bridge metadata fields.")
        path = row["Path"]
        if (
            not isinstance(path, str)
            or path.startswith("/")
            or "\\" in path
            or any(part in {"", ".", ".."} for part in path.split("/"))
            or path.rsplit("/", 1)[-1].startswith("~$")
            or BRIDGE.fullmatch(path.rsplit("/", 1)[-1]) is None
        ):
            raise ValueError("Invalid relative bridge path.")
        identity = path.casefold()
        if identity in result:
            raise ValueError("Duplicate bridge path.")
        if type(row["Bytes"]) is not int or row["Bytes"] < 0:
            raise ValueError("Bridge byte length is invalid.")
        if not isinstance(row["ETag"], str) or not row["ETag"]:
            raise ValueError("Bridge version metadata is missing.")
        modified = datetime.fromisoformat(row["ModifiedAtUtc"])
        if modified.utcoffset() is None:
            raise ValueError("Bridge metadata needs an explicit timezone.")
        result[identity] = {
            "Path": identity,
            "Bytes": row["Bytes"],
            "ModifiedAtUtc": modified.astimezone(UTC).isoformat(),
            "ETag": row["ETag"],
        }
    return result


def metadata_fingerprint(rows: Any) -> str:
    canonical = metadata_inventory(rows)
    payload = json.dumps(
        [canonical[key] for key in sorted(canonical)],
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def checked_snapshot(value: Any) -> dict[str, Any]:
    if (
        not isinstance(value, dict)
        or type(value.get("SchemaVersion")) is not int
        or value["SchemaVersion"] != 1
        or value.get("SourceRoot") != "Files/Demand"
    ):
        raise ValueError("Invalid hourly snapshot identity.")
    captured = datetime.fromisoformat(value["CapturedAtUtc"])
    if captured.utcoffset() is None:
        raise ValueError("Hourly snapshot timestamp needs a timezone.")
    if value.get("Fingerprint") != metadata_fingerprint(value.get("Files")):
        raise ValueError("Hourly snapshot fingerprint does not match its inventory.")
    return value


def parse_hourly_manifest(text: str) -> dict[str, Any]:
    if not isinstance(text, str) or not text:
        raise ValueError(
            "HOURLY mode requires FM_METADATA_JSON from the Functions activity."
        )
    value = json.loads(text)
    if (
        not isinstance(value, dict)
        or type(value.get("SchemaVersion")) is not int
        or value["SchemaVersion"] != 1
        or value.get("Status") not in {"BASELINE_REQUIRED", "CHANGED", "UNCHANGED"}
        or type(value.get("NeedsRefresh")) is not bool
        or value.get("WorkbookContentsRead") is not False
    ):
        raise ValueError("Expected the successful metadata detector output.")
    snapshot = checked_snapshot(value.get("Snapshot"))
    if value.get("CurrentFiles") != len(snapshot["Files"]):
        raise ValueError("Hourly inventory count does not reconcile.")
    return snapshot


def hourly_changes(snapshot: dict[str, Any], baseline: Any) -> list[str] | None:
    """Recheck under the shared writer lease; None means another run handled it."""
    current = metadata_inventory(snapshot["Files"])
    previous: dict[str, dict[str, Any]] = {}
    if baseline is not None:
        checked_snapshot(baseline)
        if baseline.get("Status") != "SUCCESS" or not baseline.get("RefreshRunID"):
            raise ValueError("The committed hourly baseline is invalid.")
        if baseline["Fingerprint"] == snapshot["Fingerprint"]:
            return None
        if datetime.fromisoformat(snapshot["CapturedAtUtc"]) <= datetime.fromisoformat(
            baseline["CapturedAtUtc"]
        ):
            raise ValueError(
                "This metadata snapshot is stale. Run the hourly pipeline again."
            )
        previous = metadata_inventory(baseline["Files"])
    original = {row["Path"].casefold(): row["Path"] for row in snapshot["Files"]}
    # Deletions still need an ALL inventory run, even when this list is empty.
    return [
        original[key] for key in sorted(current) if current[key] != previous.get(key)
    ]


def validate_processed_snapshot(engine: Any, snapshot: dict[str, Any]) -> None:
    expected = metadata_inventory(snapshot["Files"])
    found: dict[str, dict[str, Any]] = {}
    for meta in engine.discovery_metadata:
        key = meta["relative"].replace("\\", "/").casefold()
        if key in found:
            raise ValueError("Duplicate path in the refresh inventory.")
        found[key] = meta
    if set(found) != set(expected):
        raise ValueError(
            "Bridge paths changed after the metadata check; rerun the hourly pipeline."
        )
    for key, row in expected.items():
        meta = found[key]
        modified = datetime.fromisoformat(meta["modified"])
        # Spark binaryFile materializes modificationTime as a naive UTC datetime.
        if modified.utcoffset() is None:
            modified = modified.replace(tzinfo=UTC)
        if (
            meta["size"] != row["Bytes"]
            or modified.astimezone(UTC).isoformat() != row["ModifiedAtUtc"]
        ):
            raise ValueError(
                "A bridge changed after the metadata check: " + meta["relative"]
            )
    completed = {
        event["SourceRelativePath"].casefold()
        for event in engine.events
        if event["Status"] in {"METADATA_UNCHANGED", "SAME_CONTENT", "SUCCEEDED"}
    }
    if completed != set(expected):
        raise ValueError(
            "Not every inventoried bridge completed successfully; baseline retained."
        )


class HourlyWorker:
    def __init__(self, fs: Any, store: Any, state_folder: str):
        self.fs, self.store, self.state_folder = fs, store, state_folder.rstrip("/")

    def run(
        self,
        snapshot: dict[str, Any],
        refresh: Callable[[list[str]], tuple[Any, dict[str, Any]]],
    ) -> tuple[Any, dict[str, Any]]:
        snapshot = checked_snapshot(snapshot)
        if not self.store.acquire():
            raise WorkerRefreshError(
                "Another refresh owns the writer lock. Retry the hourly pipeline after it finishes."
            )
        batch_id, started = str(uuid4()), queue_utc()
        engine: Any = None
        summary: dict[str, Any] = {}
        try:
            changed = hourly_changes(snapshot, self.store.read("metadata-success.json"))
            if changed is None:
                return None, {
                    "Status": "IDLE",
                    "Mode": "HOURLY",
                    "Message": "This inventory was already refreshed successfully.",
                }
            for folder in ("hourly", "metadata-baselines"):
                if not self.fs.mkdirs(self.state_folder + "/" + folder):
                    raise RuntimeError("Cannot prepare hourly receipt folders.")
            print(
                "Hourly metadata changes requiring content checks:",
                len(changed),
                flush=True,
            )
            engine, summary = refresh(changed)
            if (
                summary.get("Status") != "SUCCESS"
                or summary.get("StageErrors")
                or summary.get("FileResults", {}).get("FAILED", 0)
                or summary.get("ModelRefresh", {}).get("Status")
                not in {"COMPLETED", "UNCHANGED"}
                or not summary.get("RunID")
            ):
                raise WorkerRefreshError(
                    "Ingestion, reporting or model refresh did not succeed; hourly baseline retained."
                )
            validate_processed_snapshot(engine, snapshot)
            baseline = {
                **snapshot,
                "Status": "SUCCESS",
                "RefreshRunID": summary["RunID"],
                "BatchID": batch_id,
                "RecordedAtUtc": queue_utc(),
            }
            result = {
                "SchemaVersion": 1,
                "CodeVersion": "R05.3",
                "Mode": "HOURLY",
                "Status": "SUCCESS",
                "BatchID": batch_id,
                "StartedAtUtc": started,
                "FinishedAtUtc": queue_utc(),
                "Refresh": summary,
                "MetadataFingerprint": snapshot["Fingerprint"],
                "CurrentFiles": len(snapshot["Files"]),
                "ContentChecksRequested": len(changed),
                "BaselineReceipt": "metadata-baselines/" + batch_id + ".json",
            }
            # Retain the full successful metadata before replacing its current pointer.
            self.store.write(result["BaselineReceipt"], baseline)
            self.store.write("hourly/" + batch_id + ".json", result)
            self.store.write("metadata-success.json", baseline)
            self.store.write("latest.json", result)
            return engine, result
        except Exception as exc:
            failure = {
                "SchemaVersion": 1,
                "CodeVersion": "R05.3",
                "Mode": "HOURLY",
                "Status": "FAILED",
                "BatchID": batch_id,
                "StartedAtUtc": started,
                "FinishedAtUtc": queue_utc(),
                "Error": str(exc),
                "Refresh": summary,
            }
            # A separate failure receipt preserves any earlier successful-stage receipt.
            self.store.write("hourly/failed-" + batch_id + ".json", failure)
            raise
        finally:
            self.store.release()

# Verify the destination and check the queue before loading the ingestion code.
import notebookutils

worker_context = notebookutils.runtime.context
if (str(worker_context.get("defaultLakehouseId", "")).lower() != LAKEHOUSE_ID.lower()
    or str(worker_context.get("defaultLakehouseWorkspaceId", "")).lower() != WORKSPACE_ID.lower()):
    raise RuntimeError("Attach ForecastMaster as this notebook's default lakehouse, then run again.")
worker_files_root = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}/Files"
worker_store = OneLakeQueueStore(WORKSPACE_ID, LAKEHOUSE_ID, notebookutils.credentials.getToken)
fm_mode = str(FM_RUN_MODE).upper()
if fm_mode not in {"REQUESTS", "HOURLY"}:
    raise ValueError("FM_RUN_MODE must be REQUESTS or HOURLY.")
if fm_mode == "HOURLY":
    fm_hourly_snapshot = parse_hourly_manifest(FM_METADATA_JSON)
    hourly_worker = HourlyWorker(notebookutils.fs, worker_store, worker_files_root + "/ForecastMasterRefreshState")
    print("Hourly mode: validated complete metadata snapshot for", len(fm_hourly_snapshot["Files"]), "bridges.", flush=True)
else:
    request_worker = RequestWorker(
        notebookutils.fs, worker_store,
        worker_files_root + "/Demand/ForecastMaster Refresh Requests",
        worker_files_root + "/ForecastMasterRefreshState",
    )
    waiting = request_worker.pending()
    print("New refresh requests:", len(waiting), flush=True)
    if not waiting:
        print("IDLE: no new request; no bridge discovery or model refresh ran.", flush=True)
        notebookutils.notebook.exit('{"Status":"IDLE","PendingRequests":0}')


In [ ]:
"""Read current V18 saved bridge values without an intermediate Excel Master."""

from __future__ import annotations

import hashlib
import io
import json
import math
import posixpath
import re
from collections.abc import Iterator, Mapping, Sequence
from datetime import date
from typing import TYPE_CHECKING, Any
from urllib.parse import unquote
from zipfile import ZipFile

from defusedxml.ElementTree import fromstring, iterparse

if TYPE_CHECKING:
    from xml.etree.ElementTree import (
        # annotation only; all parsing uses defusedxml.
        Element,  # nosec B405
    )

CONTRACT = "V18-explicit-profile-and-context-v2|legacy-metadata-compat-v1"
NS = {"s": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
RID = "{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id"
IDS = (
    "Channel",
    "Sub Channel",
    "Sector",
    "Manufacturer",
    "Product",
    "Section",
    "Measure",
)
MONTHS = (
    "Jan",
    "Feb",
    "Mar",
    "Apr",
    "May",
    "Jun",
    "Jul",
    "Aug",
    "Sep",
    "Oct",
    "Nov",
    "Dec",
)
LONG_MONTHS = (
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December",
)
SECTION_ALIASES = {
    "Ex-Factory": "Sub Channel - Exfactory",
    "Ex-Factory - New Contracts": "Sub Channel - Exfactory - New Contracts",
    "Ex-Factory - Shipment Adjustments": "Sub Channel - Exfactory - Shipment Adjustments",
}
WORKDAYS = {"Working days in month", "Working days in month (value)", "Working days"}


def measure_key(section: str, measure: str) -> str:
    """Keep source measure identity independent of its editable display label."""
    value = json.dumps([section, measure], ensure_ascii=False, separators=(",", ":"))
    return hashlib.sha256(value.encode()).hexdigest()


def folder_identity(relative_path: str) -> tuple[str, str]:
    """Use the agreed filename cycle anywhere under Demand; folders are not admission rules."""
    filename = unquote(relative_path).replace("\\", "/").rsplit("/", 1)[-1]
    month = r"(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec) \d{2}"
    match = re.fullmatch(
        rf"[^_]+_.+?_Bridge_(?P<cycle>{month})_EF-{month}_IMS-{month}_.+\.(?:xlsx|xlsm)",
        filename,
        re.IGNORECASE,
    )
    if match is None or filename.startswith("~$"):
        raise ValueError("Filename is outside the agreed bridge naming convention")
    label, year = match["cycle"].split(" ")
    number = next(
        i for i, name in enumerate(MONTHS, 1) if name.casefold() == label.casefold()
    )
    return f"{2000 + int(year):04d}-{number:02d}", ""


def folder_cycle(relative_path: str) -> str:
    """Compatibility name: the cycle now comes from the filename, never a folder."""
    return folder_identity(relative_path)[0]


def relationships(package: ZipFile, owner: str) -> dict[str, str]:
    """Resolve the workbook's internal sheet and table parts."""
    folder, name = posixpath.split(owner)
    path = f"{folder}/_rels/{name}.rels"
    if path not in package.namelist():
        return {}
    return {
        node.attrib["Id"]: node.attrib["Target"].lstrip("/")
        if node.attrib["Target"].startswith("/")
        else posixpath.normpath(posixpath.join(folder, node.attrib["Target"]))
        for node in fromstring(package.read(path))
        if node.attrib.get("TargetMode") != "External"
    }


def column_number(address: str) -> int:
    """Translate an OOXML cell address to a one-based column."""
    result = 0
    for char in "".join(c for c in address if c.isalpha()).upper():
        result = result * 26 + ord(char) - 64
    return result


def cached_value(cell: Element | None, strings: Sequence[str]) -> dict[str, Any]:
    """Preserve saved cache type, formula, blank and true numeric zero."""
    node = cell.find("s:v", NS) if cell is not None else None
    formula = cell.find("s:f", NS) if cell is not None else None
    value = node.text if node is not None else None
    result: dict[str, Any] = {
        "kind": "blank",
        "raw": None,
        "number": None,
        "text": None,
        "boolean": None,
        "formula": None if formula is None else (formula.text or ""),
        "cache_state": "absent"
        if node is None
        else "empty"
        if value in {None, ""}
        else "present",
    }
    kind = cell.get("t", "n") if cell is not None else "n"
    if kind == "e":
        raise ValueError("Cached Excel error: " + str(value))
    if kind == "inlineStr" and cell is not None:
        inline = cell.find("s:is", NS)
        value = "".join(inline.itertext()) if inline is not None else ""
        kind = "str"
    if value in {None, ""}:
        return result
    return _typed_cache(result, kind, str(value), strings)


def _typed_cache(
    result: dict[str, Any], kind: str, value: str, strings: Sequence[str]
) -> dict[str, Any]:
    """Decode a nonempty saved value without broad null/zero coercion."""
    if kind in {"s", "str", "d"}:
        text = strings[int(value)] if kind == "s" else value
        if text:
            result.update(kind="date" if kind == "d" else "text", raw=text, text=text)
    elif kind == "b":
        if value not in {"0", "1"}:
            raise ValueError("Invalid cached boolean")
        result.update(kind="bool", raw=value, boolean=value == "1")
    else:
        number = float(value)
        if not math.isfinite(number):
            raise ValueError("Nonfinite cached number")
        result.update(kind="number", raw=value, number=number)
    return result


def explicit_metadata(book: Element, cycle: str, folder_profile: str) -> dict[str, str]:
    """Read saved identities; normalize legacy profile tags while retaining original names."""
    definitions = {
        n.attrib["name"]: n.text or ""
        for n in book.findall("s:definedNames/s:definedName", NS)
    }
    named = {}
    for key, value in definitions.items():
        literal = value.removeprefix("=")
        if (
            key.startswith("Bridge_")
            and literal.startswith('"')
            and literal.endswith('"')
        ):
            named[key.removeprefix("Bridge_")] = literal[1:-1].replace('""', '"')
    required = (
        "Country",
        "GrowthDriver",
        "Cycle",
        "BridgeRevision",
        "SourceProfile",
        "PublicationStatus",
        "ExFactoryActualsThrough",
        "IMSActualsThrough",
    )
    missing = [key for key in required if not named.get(key)]
    if missing:
        raise ValueError("Missing explicit workbook metadata: " + ", ".join(missing))
    raw_profile = named["SourceProfile"]
    legacy = re.fullmatch(r"V[0-9]+(?P<ext>_Ext)?", raw_profile, re.IGNORECASE)
    if legacy is not None:
        named["SourceProfile"] = "Ext" if legacy["ext"] else "Local"
        if "Bridge_PublicationOrigin" not in definitions:
            # Explicitly unavailable; this does not manufacture publication provenance.
            named["PublicationOrigin"] = "NOT_RECORDED"
    if not named.get("PublicationOrigin"):
        raise ValueError("Missing explicit workbook metadata: PublicationOrigin")
    if named["Cycle"] != cycle or named["SourceProfile"] not in {"Local", "Ext"}:
        raise ValueError("Workbook cycle/profile differs from source scope")
    if folder_profile and named["SourceProfile"] != folder_profile:
        raise ValueError("Workbook profile differs from V18/V18_Ext subfolder")
    for key in ("ExFactoryActualsThrough", "IMSActualsThrough"):
        date.fromisoformat(named[key])
        if named[key][:7] >= cycle:
            raise ValueError("Actuals cutoff is not before the forecast cycle")
    version = {"DEVELOPMENT": "Dev", "PUBLISHED": "ADS"}.get(
        named["PublicationStatus"], "UNKNOWN"
    )
    return {
        **named,
        "DemandCycle": cycle,
        "Version": version,
        "Revision": named["BridgeRevision"],
        "PreviousPublication": named.get("PreviousPublication", ""),
        "DefinedNamesJson": json.dumps(definitions, ensure_ascii=False),
        "Contract": CONTRACT,
    }


def context_role(section: str, measure: str) -> str:
    """Keep the upgraded Master's calendar, controls and saved quality observable."""
    if measure in WORKDAYS:
        return "Calendar"
    if measure == "Residual Product" or (
        section
        in {"Ex-Factory - New Contracts", "Sub Channel - Exfactory - New Contracts"}
        and re.fullmatch(r"Contract \d+ (Litres|Pounds)", measure)
    ):
        return "Model Control"
    if section == "Product reconciliation checks" or (section, measure) == (
        "Pricing",
        "Status",
    ):
        return "Data Quality"
    return "Observation"


class BridgeReader:
    """Stream the current integrated source tables from one saved bridge."""

    def __init__(self, content: bytes, relative_path: str) -> None:
        """Keep source bytes and scope separate from business publication approval."""
        self.content = content
        self.relative_path = unquote(relative_path)
        self.sha256 = hashlib.sha256(content).hexdigest()
        self.cycle, self.folder_profile = folder_identity(relative_path)
        self.metadata: dict[str, str] = {}
        self.row_count = 0
        self.observation_count = 0
        self.type_counts: dict[str, int] = {}
        self.calendar: tuple[float | None, ...] | None = None

    def rows(self) -> Iterator[dict[str, Any]]:
        """Read every labelled row and month; never apply a fixed measure allowlist."""
        with ZipFile(io.BytesIO(self.content)) as package:
            book = fromstring(package.read("xl/workbook.xml"))
            self.metadata = explicit_metadata(book, self.cycle, self.folder_profile)
            strings = self._strings(package)
            parts = relationships(package, "xl/workbook.xml")
            seen: set[tuple[str, ...]] = set()
            horizon: list[str] | None = None
            tables = 0
            for sheet in book.findall("s:sheets/s:sheet", NS):
                part = parts[sheet.attrib[RID]]
                table = self._model_table(package, part)
                if table is None:
                    continue
                tables += 1
                headers = [
                    n.attrib["name"]
                    for n in table.findall("s:tableColumns/s:tableColumn", NS)
                ]
                dates = _month_dates(headers)
                if horizon is not None and horizon != headers[7:]:
                    raise ValueError("Sector month horizons differ")
                horizon = headers[7:]
                yield from self._sheet_rows(
                    package,
                    part,
                    table,
                    sheet.attrib["name"],
                    headers,
                    dates,
                    strings,
                    seen,
                )
            if not tables or self.calendar is None:
                raise ValueError(
                    "Missing SectorModel tables or source working-day calendar"
                )

    @staticmethod
    def _strings(package: ZipFile) -> list[str]:
        """Read shared strings with their original text."""
        if "xl/sharedStrings.xml" not in package.namelist():
            return []
        return [
            "".join(n.itertext())
            for n in fromstring(package.read("xl/sharedStrings.xml")).findall(
                "s:si", NS
            )
        ]

    @staticmethod
    def _model_table(package: ZipFile, part: str) -> Element | None:
        """Locate one integrated sector table without depending on sheet order."""
        models = []
        for target in relationships(package, part).values():
            if target.startswith("xl/tables/") and target.endswith(".xml"):
                table = fromstring(package.read(target))
                if re.fullmatch(r"SectorModel\d+", table.attrib.get("name", "")):
                    models.append(table)
        if len(models) > 1:
            raise ValueError("Expected one integrated table per sector")
        return models[0] if models else None

    def _sheet_rows(
        self,
        package: ZipFile,
        part: str,
        table: Element,
        sheet: str,
        headers: list[str],
        dates: list[str],
        strings: list[str],
        seen: set[tuple[str, ...]],
    ) -> Iterator[dict[str, Any]]:
        """Stream saved rows within the named table bounds."""
        first, last = table.attrib["ref"].split(":")
        first_col = column_number(first)
        first_row, last_row = (int(re.sub(r"\D", "", p)) for p in (first, last))
        if column_number(last) - first_col + 1 != len(headers):
            raise ValueError("Table width differs from headers")
        with package.open(part) as stream:
            for _, row in iterparse(stream, events=("end",)):
                if row.tag != "{" + NS["s"] + "}row":
                    continue
                ordinal = int(row.attrib["r"])
                if first_row < ordinal <= last_row:
                    cells = {column_number(c.attrib["r"]): c for c in row}
                    values = [
                        cached_value(cells.get(first_col + i), strings)
                        for i in range(len(headers))
                    ]
                    ids = tuple(
                        ""
                        if c["kind"] == "blank"
                        else str(c["text"] if c["kind"] == "text" else c["raw"])
                        for c in values[:7]
                    )
                    if ids[5] or ids[6]:
                        _check_identity(ids, sheet, seen)
                        months = self._months(headers[7:], dates, values[7:])
                        self._check_context(ids, values[7:])
                        self.row_count += 1
                        yield {
                            "sheet": sheet,
                            "table": table.attrib["name"],
                            "row": ordinal,
                            "ids": ids,
                            "months": months,
                            "metadata": self.metadata,
                            "context_role": context_role(ids[5], ids[6]),
                        }
                row.clear()

    def _months(
        self, labels: list[str], dates: list[str], values: list[dict[str, Any]]
    ) -> list[tuple[Any, ...]]:
        """Expand explicit month observations while retaining every null month."""
        result = []
        for label, day, value in zip(labels, dates, values, strict=True):
            result.append(
                (
                    day,
                    label,
                    value["kind"],
                    value["raw"],
                    value["number"],
                    value["text"],
                    value["boolean"],
                    value["formula"],
                    value["cache_state"],
                )
            )
            self.observation_count += 1
            kind = value["kind"]
            self.type_counts[kind] = self.type_counts.get(kind, 0) + 1
        return result

    def _check_context(
        self, ids: tuple[str, ...], values: list[dict[str, Any]]
    ) -> None:
        """Validate source calendar equality and structural residual selectors."""
        if ids[6] == "Residual Product" and any(v["kind"] != "blank" for v in values):
            raise ValueError("Residual selector gained monthly values")
        if ids[6] in WORKDAYS:
            if any(v["kind"] not in {"number", "blank"} for v in values):
                raise ValueError("Working-day calendar contains a nonnumeric value")
            current = tuple(v["number"] for v in values)
            if self.calendar is not None and self.calendar != current:
                raise ValueError("Repeated source working days differ")
            self.calendar = current


def _check_identity(
    ids: tuple[str, ...], sheet: str, seen: set[tuple[str, ...]]
) -> None:
    """Reject incomplete or duplicate business identities instead of summing them."""
    if not ids[5] or not ids[6] or ids[2] != sheet or ids in seen:
        raise ValueError("Incomplete, wrong-sector or duplicate bridge identity")
    seen.add(ids)


def _month_dates(headers: list[str]) -> list[str]:
    """Accept an extended continuous horizon with the exact seven identity columns."""
    if tuple(headers[:7]) != IDS:
        raise ValueError("Source layout needs an adapter")
    dates = []
    for header in headers[7:]:
        match = re.fullmatch(r"(\d{4})-([A-Za-z]{3})", header)
        if not match or match[2] not in MONTHS:
            raise ValueError("Unknown model month header: " + header)
        dates.append(date(int(match[1]), MONTHS.index(match[2]) + 1, 1).isoformat())
    ordinal = [int(d[:4]) * 12 + int(d[5:7]) for d in dates]
    if not ordinal or ordinal != list(range(ordinal[0], ordinal[0] + len(ordinal))):
        raise ValueError("Model month horizon is not continuous")
    return dates


def normalize_catalog(rows: Sequence[Mapping[str, Any]]) -> list[dict[str, str]]:
    """Validate user-editable definitions without changing original measure identities."""
    normalized = []
    intervals: dict[str, list[tuple[str, str]]] = {}
    for original in rows:
        section, measure = original.get("SourceSection"), original.get("SourceMeasure")
        if (
            not isinstance(section, str)
            or not isinstance(measure, str)
            or not section
            or not measure
        ):
            raise ValueError("Catalog needs exact SourceSection and SourceMeasure")
        row = {k: "" if v is None else str(v) for k, v in original.items()}
        key = measure_key(section, measure)
        enabled = row.get("ReportingEnabled", "true").lower()
        if enabled not in {"true", "false"}:
            raise ValueError("ReportingEnabled must be true or false")
        lower, upper = _effective_range(row)
        if any(max(lower, a) <= min(upper, b) for a, b in intervals.get(key, [])):
            raise ValueError("Catalog effective ranges overlap")
        intervals.setdefault(key, []).append((lower, upper))
        row.update(
            MeasureKey=key,
            ReportingEnabled=enabled,
            EffectiveFromCycle=lower,
            EffectiveToCycle=upper,
            DisplayName=row.get("DisplayName") or measure,
            RawConfigurationJson=json.dumps(
                dict(original), ensure_ascii=False, sort_keys=True, default=str
            ),
        )
        normalized.append(row)
    return normalized


def _effective_range(row: Mapping[str, str]) -> tuple[str, str]:
    """Require real, ordered year/month bounds for a reporting definition."""
    result = []
    for field, fallback in (
        ("EffectiveFromCycle", "0001-01"),
        ("EffectiveToCycle", "9999-12"),
    ):
        value = row.get(field) or fallback
        if not re.fullmatch(r"\d{4}-\d{2}", value):
            raise ValueError("Invalid effective cycle")
        date(int(value[:4]), int(value[5:]), 1)
        result.append(value)
    if result[0] > result[1]:
        raise ValueError("Effective cycle range is reversed")
    return result[0], result[1]


WIDE_FIELDS = (
    "SourceRevisionID",
    "SourceFileName",
    "SourceRelativePath",
    "SourceSheet",
    "SourceTable",
    "MeasureKey",
    "Country",
    "GrowthDriver",
    "DemandCycle",
    "Version",
    "Revision",
    "ExFactoryActualsThrough",
    "IMSActualsThrough",
    "PublicationStatus",
    "PublicationOrigin",
    "SourceProfile",
    "PreviousPublication",
    "CanonicalSection",
    "ContextRole",
    "Channel",
    "SubChannel",
    "Sector",
    "Manufacturer",
    "Product",
    "Section",
    "Measure",
)
REVISION_FIELDS = (
    "SourceRevisionID",
    "SourceBytes",
    "SourceFileName",
    "SourceRelativePath",
    "Contract",
    "FirstImportedAtUtc",
    "WideRows",
    "Observations",
    "DefinedNamesJson",
    "PublicationStatus",
    "PublicationOrigin",
    "SourceProfile",
    "Country",
    "GrowthDriver",
    "DemandCycle",
    "Revision",
    "ExFactoryActualsThrough",
    "IMSActualsThrough",
    "PreviousPublication",
)


def history_record(
    row: Mapping[str, Any], source_id: str, filename: str, relative: str
) -> dict[str, Any]:
    """Bind one compact parsed row to the named Spark boundary schema."""
    meta, ids = row["metadata"], row["ids"]
    result = {
        name: meta[name]
        for name in (
            "Country",
            "GrowthDriver",
            "DemandCycle",
            "Version",
            "Revision",
            "ExFactoryActualsThrough",
            "IMSActualsThrough",
            "PublicationStatus",
            "PublicationOrigin",
            "SourceProfile",
            "PreviousPublication",
        )
    }
    result.update(
        zip(
            (
                "Channel",
                "SubChannel",
                "Sector",
                "Manufacturer",
                "Product",
                "Section",
                "Measure",
            ),
            ids,
            strict=True,
        )
    )
    result.update(
        SourceRevisionID=source_id,
        SourceFileName=filename,
        SourceRelativePath=relative,
        SourceSheet=row["sheet"],
        SourceTable=row["table"],
        SourceRow=row["row"],
        Months=row["months"],
        MeasureKey=measure_key(ids[5], ids[6]),
        CanonicalSection=SECTION_ALIASES.get(ids[5], ids[5]),
        ContextRole=row["context_role"],
    )
    return result


def revision_record(reader: BridgeReader, captured_at: str) -> dict[str, str]:
    """Keep explicit publication provenance and profile in the completion receipt."""
    meta = reader.metadata
    result = {
        name: meta[name]
        for name in (
            "Country",
            "GrowthDriver",
            "DemandCycle",
            "Revision",
            "ExFactoryActualsThrough",
            "IMSActualsThrough",
            "PublicationStatus",
            "PublicationOrigin",
            "SourceProfile",
            "PreviousPublication",
            "DefinedNamesJson",
            "Contract",
        )
    }
    result.update(
        SourceRevisionID=reader.sha256,
        SourceBytes=str(len(reader.content)),
        SourceFileName=reader.relative_path.rsplit("/", 1)[-1],
        SourceRelativePath=reader.relative_path,
        FirstImportedAtUtc=captured_at,
        WideRows=str(reader.row_count),
        Observations=str(reader.observation_count),
    )
    return result


In [ ]:
"""Changed-file Fabric ingestion. The notebook embeds this module and the unchanged value decoder."""

from __future__ import annotations

import hashlib
import json
import re
import time
import uuid
from collections import Counter
from collections.abc import Iterator
from contextlib import contextmanager
from datetime import UTC, datetime
from typing import Any
from urllib.parse import unquote

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql import types as T

FILE_MONTH = r"(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec) \d{2}"
NAME_PATTERN = re.compile(
    rf"[^_]+_.+?_Bridge_{FILE_MONTH}_EF-{FILE_MONTH}_IMS-{FILE_MONTH}_.+\.(?:xlsx|xlsm)",
    re.IGNORECASE,
)
FILE_GLOB = "*_[Bb][Rr][Ii][Dd][Gg][Ee]_*_[Ee][Ff]-*_[Ii][Mm][Ss]-*.[Xx][Ll][Ss]*"
EVENT_FIELDS = [
    "RunID",
    "ObservedAtUtc",
    "SourcePath",
    "SourceRelativePath",
    "SourceModifiedAt",
    "SourceBytes",
    "ReaderContract",
    "Status",
    "SourceRevisionID",
    "Observations",
    "Message",
]


def filename_candidate(name: str) -> bool:
    """Any version suffix and any folder; only the agreed xlsx/xlsm filename contract."""
    return not name.startswith("~$") and NAME_PATTERN.fullmatch(name) is not None


def normalized_path(path: str) -> str:
    """SharePoint paths are case insensitive; separators do not create scenarios."""
    return "/".join(p for p in path.replace("\\", "/").split("/") if p).casefold()


def report_identity(content_id: str, path: str, original_path: str = "") -> str:
    """One permanent report identity per saved file, independent of edited bytes."""
    if not path:
        raise ValueError("A saved file needs an explicit relative path")
    return hashlib.sha256(
        ("current-file-v1|" + normalized_path(path)).encode()
    ).hexdigest()


def current_content_ids(
    pointers: list[dict[str, str]], completed: set[str]
) -> list[str]:
    """Do not delete any dataset still referenced by a complete file pointer."""
    active = {row["SourceRevisionID"] for row in pointers}
    if not active or not active <= completed:
        raise ValueError("Cannot compact an incomplete current-file index")
    return sorted(active)


def utc_now() -> str:
    return datetime.now(UTC).isoformat()


class FastRefresh:
    """One writer. Events are batched; each changed workbook is materialized and merged once."""

    def __init__(self, spark: Any, reader: dict[str, Any], config: dict[str, Any]):
        self.spark, self.reader, self.config = spark, reader, config
        self.prefix = config["TABLE_PREFIX"]
        if not re.fullmatch(r"[a-z][a-z0-9_]*", self.prefix):
            raise ValueError("Invalid table prefix")
        self.allowed = set(config["ALLOWED_PROFILES"])
        if not self.allowed or not self.allowed <= {"Local", "Ext"}:
            raise ValueError("Choose Local and/or Ext explicitly")
        self.root = config["SOURCE_ROOT"].rstrip("/")
        self.run_id = str(uuid.uuid4())
        self.contract = (
            reader["CONTRACT"] + "|affiliate-bridge-cycle-ef-ims-any-suffix-v1"
        )
        self.events: list[dict[str, str]] = []
        self.timings: list[dict[str, Any]] = []
        self.writes: Counter[str] = Counter()
        self.state_updates: list[dict[str, str]] = []
        self.new_source_ids: list[str] = []
        self.completed: dict[str, dict[str, str]] = {}
        self.source_changed = False
        self.stage_errors: list[str] = []
        self.report_changed = False
        self.current_report_files = 0
        self.failed_files = 0
        self.discovered_paths: set[str] | None = None
        self.discovery_metadata: list[dict[str, Any]] = []
        self.run_started = time.perf_counter()

    @contextmanager
    def stage(self, name: str, filename: str = "") -> Iterator[None]:
        started = time.perf_counter()
        record = {"Stage": name, "FileName": filename, "Seconds": 0.0, "Status": "OK"}
        print(f"Starting: {name}" + (f" — {filename}" if filename else ""), flush=True)
        try:
            yield
        except Exception:
            record["Status"] = "FAILED"
            raise
        finally:
            record["Seconds"] = round(time.perf_counter() - started, 3)
            self.timings.append(record)
            print(f"{name}: {record['Seconds']:.2f}s ({record['Status']})", flush=True)

    def event(
        self,
        meta: dict[str, Any],
        status: str,
        source_id: str = "",
        count: int = 0,
        message: str = "",
    ) -> None:
        self.events.append(
            dict(
                zip(
                    EVENT_FIELDS,
                    [
                        self.run_id,
                        utc_now(),
                        meta["path"],
                        meta["relative"],
                        meta["modified"],
                        str(meta["size"]),
                        self.contract,
                        status,
                        source_id,
                        str(count),
                        message,
                    ],
                    strict=True,
                )
            )
        )

    def string_frame(self, rows: list[dict[str, str]], names: list[str]) -> Any:
        schema = T.StructType([T.StructField(n, T.StringType(), True) for n in names])
        return self.spark.createDataFrame(rows, schema)

    def upsert(
        self,
        name: str,
        frame: Any,
        keys: list[str],
        *,
        source_id: str = "",
        update: bool = False,
    ) -> None:
        """Content tables insert missing keys; mutable metadata pointers may update."""
        if not self.spark.catalog.tableExists(name):
            frame.coalesce(2).write.format("delta").mode("errorifexists").saveAsTable(
                name
            )
        else:
            condition = " AND ".join(f"target.`{k}` <=> source.`{k}`" for k in keys)
            if source_id:
                condition += " AND target.SourceRevisionID = '" + source_id + "'"
            merge = (
                DeltaTable.forName(self.spark, name)
                .alias("target")
                .merge(frame.alias("source"), condition)
            )
            if update:
                merge = merge.whenMatchedUpdateAll()
            merge.whenNotMatchedInsertAll().execute()
        self.writes[name] += 1

    def replace_snapshot(self, name: str, frame: Any, keys: list[str]) -> None:
        """Atomically replace a derived/configuration snapshot using its declared unique keys."""
        if not self.spark.catalog.tableExists(name):
            frame.coalesce(1).write.format("delta").mode("errorifexists").saveAsTable(
                name
            )
        else:
            condition = " AND ".join(f"target.`{k}` <=> source.`{k}`" for k in keys)
            (
                DeltaTable.forName(self.spark, name)
                .alias("target")
                .merge(frame.alias("source"), condition)
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .whenNotMatchedBySourceDelete()
                .execute()
            )
        self.writes[name] += 1

    def _metadata(self, row: Any) -> dict[str, Any]:
        path = row["path"]
        decoded = unquote(path)
        marker = "/" + self.root.strip("/") + "/"
        if marker not in decoded:
            raise ValueError(
                "Discovered source is outside the configured Demand root: " + decoded
            )
        relative = decoded.split(marker, 1)[1]
        return {
            "path": path,
            "relative": relative,
            "name": relative.rsplit("/", 1)[-1],
            "size": int(row["length"]),
            "modified": row["modificationTime"].isoformat(),
            "modified_value": row["modificationTime"],
        }

    def discover(self) -> list[dict[str, Any]]:
        """Full discovery finds new folders. FILES mode checks exact known paths without a root crawl."""
        mode = self.config.get("REFRESH_MODE", "ALL").upper()
        paths = self.config.get("SELECTED_RELATIVE_PATHS", [])
        if mode == "FILES":
            if not paths or any(
                p.startswith("/") or ":" in p or ".." in p.split("/") for p in paths
            ):
                raise ValueError("FILES mode needs exact relative paths inside Demand")
            roots = [self.root + "/" + p for p in dict.fromkeys(paths)]
        elif mode == "ALL":
            roots = self.root
        else:
            raise ValueError("REFRESH_MODE must be ALL or FILES")
        rows = (
            self.spark.read.format("binaryFile")
            .option("recursiveFileLookup", True)
            .option("pathGlobFilter", FILE_GLOB)
            .load(roots)
            .select("path", "length", "modificationTime")
            .collect()
        )
        candidates = []
        for row in rows:
            meta = self._metadata(row)
            if filename_candidate(meta["name"]):
                candidates.append(meta)
            else:
                self.event(meta, "OUTSIDE_FILENAME_PATTERN")
        print(
            f"Files inspected: {len(rows)}; matching bridge filenames: {len(candidates)}",
            flush=True,
        )
        if not candidates:
            raise ValueError(
                "No files match the agreed bridge filename convention in this scope"
            )
        return sorted(candidates, key=lambda x: x["path"])

    def load_state(self) -> dict[str, tuple[str, str, str]]:
        revisions = self.prefix + "_source_revisions"
        if self.spark.catalog.tableExists(revisions):
            self.completed = {
                r["SourceRevisionID"]: r.asDict()
                for r in self.spark.table(revisions).collect()
            }
        state_name = self.prefix + "_file_state"
        if self.spark.catalog.tableExists(state_name):
            return {
                r["SourcePath"]: (
                    r["SourceModifiedAt"],
                    r["SourceBytes"],
                    r["SourceRevisionID"],
                )
                for r in self.spark.table(state_name)
                .where(F.col("ReaderContract") == self.contract)
                .collect()
            }
        events = self.prefix + "_ingestion_events"
        if not self.spark.catalog.tableExists(events):
            return {}
        rows = (
            self.spark.table(events)
            .where(F.col("Status").isin("SUCCEEDED", "SAME_CONTENT"))
            .where(F.col("ReaderContract") == self.contract)
            .groupBy("SourcePath")
            .agg(
                F.max(
                    F.struct(
                        "ObservedAtUtc",
                        "SourceModifiedAt",
                        "SourceBytes",
                        "SourceRevisionID",
                    )
                ).alias("last")
            )
            .collect()
        )
        return {
            r["SourcePath"]: (
                r["last"]["SourceModifiedAt"],
                r["last"]["SourceBytes"],
                r["last"]["SourceRevisionID"],
            )
            for r in rows
        }

    def remember(self, meta: dict[str, Any], source_id: str) -> None:
        self.state_updates.append(
            {
                "SourcePath": meta["path"],
                "SourceRelativePath": meta["relative"],
                "SourceModifiedAt": meta["modified"],
                "SourceBytes": str(meta["size"]),
                "SourceRevisionID": source_id,
                "ReaderContract": self.contract,
                "LastSuccessAtUtc": utc_now(),
            }
        )

    def ingest_one(
        self, meta: dict[str, Any], previous: dict[str, tuple[str, str, str]]
    ) -> None:
        prior = previous.get(meta["path"])
        if (
            not self.config.get("FORCE_REHASH", False)
            and normalized_path(meta["relative"])
            not in {
                normalized_path(path)
                for path in self.config.get("FORCE_REHASH_PATHS", [])
            }
            and prior
            and prior[:2] == (meta["modified"], str(meta["size"]))
            and prior[2] in self.completed
            and self.completed[prior[2]]["SourceProfile"] in self.allowed
        ):
            self.event(meta, "METADATA_UNCHANGED", prior[2])
            self.remember(meta, prior[2])
            return
        if meta["size"] > self.config.get("MAX_BRIDGE_BYTES", 128 * 1024 * 1024):
            raise ValueError(
                "Matching workbook exceeds the configured per-file byte bound"
            )
        with self.stage("Read saved workbook", meta["name"]):
            found = (
                self.spark.read.format("binaryFile")
                .load(meta["path"])
                .select("content", "length", "modificationTime")
                .collect()
            )
            if (
                len(found) != 1
                or int(found[0]["length"]) != meta["size"]
                or found[0]["modificationTime"] != meta["modified_value"]
            ):
                raise ValueError(
                    "The file changed while being read; save/upload must finish before retry"
                )
            content = bytes(found[0]["content"])
            source_id = hashlib.sha256(content).hexdigest()
        if source_id in self.completed:
            receipt = self.completed[source_id]
            expected_cycle = self.reader["folder_cycle"](meta["relative"])
            if receipt["DemandCycle"] != expected_cycle:
                raise ValueError(
                    "Saved content cycle differs from the matching filename cycle"
                )
            if receipt["SourceProfile"] not in self.allowed:
                self.event(meta, "OTHER_PROFILE", source_id)
                return
            self.event(meta, "SAME_CONTENT", source_id)
            self.remember(meta, source_id)
            return
        with self.stage("Decode saved values", meta["name"]):
            reader = self.reader["BridgeReader"](content, meta["relative"])
            iterator = reader.rows()
            first = next(iterator)
            if reader.metadata["SourceProfile"] not in self.allowed:
                self.event(meta, "OTHER_PROFILE", source_id)
                return
            raw_rows = [
                self.reader["history_record"](
                    first, source_id, meta["name"], meta["relative"]
                )
            ]
            for row in iterator:
                raw_rows.append(
                    self.reader["history_record"](
                        row, source_id, meta["name"], meta["relative"]
                    )
                )
                if reader.observation_count > self.config.get(
                    "MAX_OBSERVATIONS_PER_FILE", 5_000_000
                ):
                    raise ValueError(
                        "Workbook exceeds the configured single-file materialization bound"
                    )
        with self.stage("Commit source tables", meta["name"]):
            self.write_source(raw_rows, source_id, reader.observation_count)
            receipt = self.reader["revision_record"](reader, utc_now())
            self.upsert(
                self.prefix + "_source_revisions",
                self.string_frame([receipt], list(self.reader["REVISION_FIELDS"])),
                ["SourceRevisionID"],
            )
        self.completed[source_id] = receipt
        self.new_source_ids.append(source_id)
        self.source_changed = True
        self.event(meta, "SUCCEEDED", source_id, reader.observation_count)
        self.remember(meta, source_id)

    def write_source(
        self, rows: list[dict[str, Any]], source_id: str, expected: int
    ) -> None:
        month_fields = [
            "DemandMonth",
            "SourceMonthHeader",
            "ValueKind",
            "ValueRaw",
            "ValueNumber",
            "ValueText",
            "ValueBoolean",
            "SourceFormula",
            "FormulaCacheState",
        ]
        month_types = [
            T.StringType(),
            T.StringType(),
            T.StringType(),
            T.StringType(),
            T.DoubleType(),
            T.StringType(),
            T.BooleanType(),
            T.StringType(),
            T.StringType(),
        ]
        months = T.StructType(
            [
                T.StructField(k, v, True)
                for k, v in zip(month_fields, month_types, strict=True)
            ]
        )
        schema = T.StructType(
            [T.StructField(n, T.StringType(), True) for n in self.reader["WIDE_FIELDS"]]
            + [
                T.StructField("SourceRow", T.LongType(), False),
                T.StructField("Months", T.ArrayType(months, False), False),
            ]
        )
        frame = (
            self.spark.createDataFrame(rows, schema)
            .coalesce(2)
            .select("*", F.explode("Months").alias("Observation"))
            .select("*", "Observation.*")
            .drop("Months", "Observation")
            .withColumn("DemandMonth", F.to_date("DemandMonth"))
            .withColumn(
                "ObservationID",
                F.sha2(
                    F.concat_ws(
                        "|",
                        "SourceRevisionID",
                        "SourceTable",
                        F.col("SourceRow").cast("string"),
                        F.col("DemandMonth").cast("string"),
                    ),
                    256,
                ),
            )
            .withColumn(
                "SourceMeasureID",
                F.sha2(F.concat_ws("|", "SourceRevisionID", "MeasureKey"), 256),
            )
            .persist()
        )
        try:
            if frame.count() != expected:
                raise ValueError(
                    "Materialized source observation count differs from the saved workbook"
                )
            self.upsert(
                self.prefix + "_history", frame, ["ObservationID"], source_id=source_id
            )
            measure_names = [
                "SourceRevisionID",
                "SourceMeasureID",
                "MeasureKey",
                "Country",
                "SourceProfile",
                "GrowthDriver",
                "DemandCycle",
                "Section",
                "Measure",
                "CanonicalSection",
                "ContextRole",
            ]
            self.upsert(
                self.prefix + "_source_measures",
                frame.select(*measure_names).distinct(),
                ["SourceMeasureID"],
                source_id=source_id,
            )
            calendar = (
                frame.where(F.col("ContextRole") == "Calendar")
                .select(
                    "SourceRevisionID",
                    "SourceProfile",
                    "Country",
                    "GrowthDriver",
                    "DemandCycle",
                    "DemandMonth",
                    F.dayofmonth(F.last_day("DemandMonth")).alias("CalendarDays"),
                    F.col("ValueNumber").alias("WorkingDays"),
                )
                .distinct()
                .withColumn(
                    "CalendarID",
                    F.sha2(
                        F.concat_ws(
                            "|", "SourceRevisionID", F.col("DemandMonth").cast("string")
                        ),
                        256,
                    ),
                )
            )
            self.upsert(
                self.prefix + "_calendar", calendar, ["CalendarID"], source_id=source_id
            )
            controls = (
                frame.where(F.col("ContextRole") == "Model Control")
                .select(
                    "SourceRevisionID",
                    "SourceProfile",
                    "Country",
                    "GrowthDriver",
                    "DemandCycle",
                    "SourceSheet",
                    "SourceTable",
                    "SourceRow",
                    "Channel",
                    "SubChannel",
                    "Sector",
                    "Manufacturer",
                    "Product",
                    "Section",
                    "Measure",
                )
                .distinct()
                .withColumn(
                    "ControlID",
                    F.sha2(
                        F.concat_ws(
                            "|",
                            "SourceRevisionID",
                            "SourceTable",
                            F.col("SourceRow").cast("string"),
                        ),
                        256,
                    ),
                )
                .withColumn(
                    "InputSlot", F.regexp_extract("Measure", r"Contract (\d+)", 1)
                )
            )
            roles = {r["ContextRole"] for r in rows}
            if "Model Control" in roles or not self.spark.catalog.tableExists(
                self.prefix + "_controls"
            ):
                self.upsert(
                    self.prefix + "_controls",
                    controls,
                    ["ControlID"],
                    source_id=source_id,
                )
            if "Data Quality" in roles or not self.spark.catalog.tableExists(
                self.prefix + "_quality"
            ):
                self.upsert(
                    self.prefix + "_quality",
                    frame.where(F.col("ContextRole") == "Data Quality"),
                    ["ObservationID"],
                    source_id=source_id,
                )
            saved = (
                self.spark.table(self.prefix + "_history")
                .where(F.col("SourceRevisionID") == source_id)
                .count()
            )
            if saved != expected:
                raise ValueError(
                    "Committed observation count differs; source receipt will not be published"
                )
        finally:
            frame.unpersist()

    def flush(self) -> None:
        """One audit commit per run; file-state pointers exist only for complete source receipts."""
        if self.events:
            frame = self.string_frame(self.events, EVENT_FIELDS)
            frame.coalesce(1).write.format("delta").mode("append").saveAsTable(
                self.prefix + "_ingestion_events"
            )
            self.writes[self.prefix + "_ingestion_events"] += 1
        if self.state_updates:
            # Existing metadata pointers are cheap; avoid updating them on an unchanged run.
            name = self.prefix + "_file_state"
            if (
                self.source_changed
                or any(e["Status"] == "SAME_CONTENT" for e in self.events)
                or not self.spark.catalog.tableExists(name)
            ):
                frame = self.string_frame(
                    self.state_updates, list(self.state_updates[0])
                )
                self.upsert(name, frame, ["SourcePath"], update=True)
        # A completed recursive inventory retires deleted/renamed paths. FILES
        # mode never removes paths outside its requested scope. Failed files
        # still present in the inventory keep their last successful pointer.
        name = self.prefix + "_file_state"
        if self.spark.catalog.tableExists(name):
            original = [r.asDict() for r in self.spark.table(name).collect()]
            latest: dict[str, dict[str, str]] = {}
            for row in original:
                key = normalized_path(row["SourceRelativePath"])
                if (
                    self.discovered_paths is not None
                    and key not in self.discovered_paths
                ):
                    continue
                if (
                    key not in latest
                    or row["LastSuccessAtUtc"] > latest[key]["LastSuccessAtUtc"]
                ):
                    latest[key] = row
            if len(latest) != len(original):
                if not latest:
                    raise ValueError(
                        "Current-file retirement would leave an empty index"
                    )
                self.replace_snapshot(
                    name,
                    self.string_frame(list(latest.values()), list(original[0])),
                    ["SourcePath"],
                )
                self.source_changed = True

    def ingest(self) -> None:
        with self.stage("Discover matching files"):
            candidates = self.discover()
            self.discovery_metadata = candidates
            if self.config.get("REFRESH_MODE", "ALL").upper() == "ALL":
                self.discovered_paths = {
                    normalized_path(m["relative"]) for m in candidates
                }
        with self.stage("Load compact change index"):
            previous = self.load_state()
        try:
            for meta in candidates:
                try:
                    self.ingest_one(meta, previous)
                except Exception as exc:  # noqa: BLE001 -- retain stage diagnostics and surface failure  # noqa: BLE001 -- retain attributable per-file failures and continue other sources
                    self.failed_files += 1
                    self.event(meta, "FAILED", message=str(exc))
                    print("FAILED:", meta["relative"], str(exc), flush=True)
        finally:
            with self.stage("Commit batched audit and change index"):
                self.flush()

    def report_sources(self) -> list[dict[str, str]]:
        """Only the latest complete pointer per saved path enters reporting."""
        name = self.prefix + "_file_state"
        if not self.spark.catalog.tableExists(name):
            raise ValueError("No committed current-file index is available")
        latest: dict[str, dict[str, str]] = {}
        for item in self.spark.table(name).collect():
            row = item.asDict()
            key = normalized_path(row["SourceRelativePath"])
            prior = latest.get(key)
            if prior is None or (row.get("LastSuccessAtUtc", ""), row["SourcePath"]) > (
                prior.get("LastSuccessAtUtc", ""),
                prior["SourcePath"],
            ):
                latest[key] = row
        rows = []
        for _, pointer in sorted(latest.items()):
            source_id, relative = (
                pointer["SourceRevisionID"],
                pointer["SourceRelativePath"],
            )
            raw = self.completed.get(source_id)
            if raw is None:
                raise ValueError(
                    "Current file has no completed source receipt: " + relative
                )
            if raw["SourceProfile"] not in self.allowed or not filename_candidate(
                relative.rsplit("/", 1)[-1]
            ):
                continue
            row = dict(raw)
            row.update(
                RawSourceRevisionID=source_id,
                SourceRevisionID=report_identity(source_id, relative),
                SourceRelativePath=relative,
                SourceFileName=relative.rsplit("/", 1)[-1],
                PathFirstSeenAtUtc=pointer["LastSuccessAtUtc"],
            )
            rows.append(row)
        if not rows:
            raise ValueError(
                "No complete matching current files are available for reporting"
            )
        self.current_report_files = len(rows)
        return rows

    def compact_current_sources(self) -> dict[str, int]:
        """Remove superseded full datasets after current reporting/model confirmation.

        Shared content survives while ANY saved path still points to it. Audit
        events keep hashes/status/timestamps, never a second full workbook copy.
        Delta's ordinary recovery retention is not disabled.
        """
        pointers = (
            self.spark.table(self.prefix + "_file_state")
            .select("SourceRevisionID")
            .distinct()
        )
        active = current_content_ids(
            [row.asDict() for row in pointers.collect()], set(self.completed)
        )
        signature = hashlib.sha256(json.dumps(sorted(active)).encode()).hexdigest()
        state_name = self.prefix + "_current_storage_state"
        previous = (
            self.spark.table(state_name).first()
            if self.spark.catalog.tableExists(state_name)
            else None
        )
        if previous and previous["Signature"] == signature and not self.failed_files:
            print(
                "Current-file data unchanged: no compaction scans or writes.",
                flush=True,
            )
            self.maintain_storage()
            return {}
        counts = {}
        for suffix in (
            "history",
            "source_measures",
            "calendar",
            "controls",
            "quality",
            "measure_assignments",
            "source_revisions",
        ):
            name = self.prefix + "_" + suffix
            if not self.spark.catalog.tableExists(name):
                continue
            stale = ~F.col("SourceRevisionID").isin(active)
            count = self.spark.table(name).where(stale).count()
            if count:
                DeltaTable.forName(self.spark, name).delete(stale)
                self.writes[name] += 1
            counts[name] = count
        self.completed = {sid: self.completed[sid] for sid in active}
        self.replace_snapshot(
            state_name,
            self.string_frame(
                [{"Signature": signature, "CompletedAtUtc": utc_now()}],
                ["Signature", "CompletedAtUtc"],
            ),
            ["Signature"],
        )
        self.maintain_storage()
        print(
            "Current-file compaction: removed superseded rows",
            json.dumps(counts),
            flush=True,
        )
        return counts

    def maintain_storage(self) -> None:
        """Weekly vacuum during an actual requested run; retain at least seven days.

        This never disables retention checks, starts an idle Spark session, or
        treats recovery files as report scenarios. Retry unfinished maintenance.
        """
        name = self.prefix + "_storage_maintenance"
        prior = (
            self.spark.table(name).first()
            if self.spark.catalog.tableExists(name)
            else None
        )
        if (
            prior
            and (
                datetime.now(UTC) - datetime.fromisoformat(prior["CompletedAtUtc"])
            ).total_seconds()
            < 7 * 86400
        ):
            return
        if prior is None:
            # Establish the weekly cadence without scanning fresh recovery files
            # on the one-time migration. First vacuum occurs after seven days.
            self.replace_snapshot(
                name,
                self.string_frame([{"CompletedAtUtc": utc_now()}], ["CompletedAtUtc"]),
                ["CompletedAtUtc"],
            )
            print(
                "Physical cleanup cadence initialized; seven-day recovery retention preserved.",
                flush=True,
            )
            return
        prefixes = {self.prefix, self.config.get("REPORT_PREFIX", "fm_review")}
        managed = {
            self.prefix + "_" + s
            for s in (
                "history",
                "source_measures",
                "calendar",
                "controls",
                "quality",
                "source_revisions",
                "measure_assignments",
                "file_state",
                "ingestion_events",
            )
        }
        managed.update(
            self.config.get("REPORT_PREFIX", "fm_review") + "_" + s
            for s in ("facts", "files", "reference", "projection_receipts", "series")
        )
        for table in sorted(managed):
            if not any(table.startswith(prefix + "_") for prefix in prefixes):
                raise ValueError("Unexpected maintenance table")
            if self.spark.catalog.tableExists(table):
                DeltaTable.forName(self.spark, table).vacuum(168)
        self.replace_snapshot(
            name,
            self.string_frame([{"CompletedAtUtc": utc_now()}], ["CompletedAtUtc"]),
            ["CompletedAtUtc"],
        )

    def summary(self) -> dict[str, Any]:
        status = "PARTIAL" if self.failed_files or self.stage_errors else "SUCCESS"
        return {
            "RunID": self.run_id,
            "Status": status,
            "FileResults": dict(Counter(e["Status"] for e in self.events)),
            "UpdatedFiles": sum(e["Status"] == "SUCCEEDED" for e in self.events),
            "DecodedChangedFiles": len(self.new_source_ids),
            "CurrentReportFiles": self.current_report_files,
            "StoragePolicy": "CURRENT_FILE_PER_PATH",
            "TableWrites": dict(self.writes),
            "StageErrors": self.stage_errors,
            "SecondsAfterSessionReady": round(
                time.perf_counter() - self.run_started, 3
            ),
        }


In [ ]:
"""Versioned measure catalog. Source value semantics are unchanged."""

from __future__ import annotations

import hashlib
import json
from datetime import UTC, datetime
from typing import Any

from pyspark.sql import functions as F
from pyspark.sql import types as T


def update_catalog(engine: Any) -> bool:
    spark = engine.spark
    TABLE_PREFIX = engine.prefix
    normalize_catalog = engine.reader["normalize_catalog"]
    MEASURE_CATALOG_TABLE = engine.config.get("MEASURE_CATALOG_TABLE", "")
    MEASURE_CATALOG_FILE = engine.config.get(
        "MEASURE_CATALOG_FILE", "Files/ForecastMasterControls/MeasureCatalog.csv"
    )
    CONFIGURATIONS = TABLE_PREFIX + "_measure_configuration_snapshots"
    try:
        if MEASURE_CATALOG_TABLE:
            config_source = spark.table(MEASURE_CATALOG_TABLE)
        else:
            config_source = (
                spark.read.option("header", True)
                .option("mode", "FAILFAST")
                .option("multiLine", True)
                .option("escape", chr(34))
                .csv(MEASURE_CATALOG_FILE)
            )
        input_rules = [r.asDict() for r in config_source.toLocalIterator()]
        rules = normalize_catalog(input_rules)
        configuration_id = hashlib.sha256(
            json.dumps(rules, sort_keys=True, ensure_ascii=False, default=str).encode(
                "utf-8"
            )
        ).hexdigest()
        signature = hashlib.sha256(
            json.dumps(
                [
                    configuration_id,
                    sorted(r["RawSourceRevisionID"] for r in engine.report_sources()),
                ],
                sort_keys=True,
            ).encode()
        ).hexdigest()
        state_name = TABLE_PREFIX + "_catalog_refresh_state"
        if (
            spark.catalog.tableExists(state_name)
            and spark.catalog.tableExists(TABLE_PREFIX + "_measure_assignments")
        ) and {r[0] for r in spark.table(state_name).select("Signature").collect()} == {
            signature
        }:
            print(
                "Measure catalog and completed sources unchanged: assignments reused.",
                flush=True,
            )
            return False
        known = [
            "MeasureKey",
            "SourceSection",
            "SourceMeasure",
            "DisplayName",
            "ReportingEnabled",
            "EffectiveFromCycle",
            "EffectiveToCycle",
            "ValueRole",
            "Unit",
            "AggregationRule",
            "ComparisonRule",
            "DefinitionStatus",
            "Owner",
            "ChangeReason",
            "RawConfigurationJson",
            "CanonicalSection",
            "Family",
            "Dependency",
        ]
        config_names = ["ConfigurationID", "CapturedAtUtc", *known]
        config_schema = T.StructType(
            [T.StructField(name, T.StringType(), True) for name in config_names]
        )
        captured = datetime.now(UTC).isoformat()
        config_rows = [
            [configuration_id, captured, *[r.get(name, "") for name in known]]
            for r in rules
        ]
        if not spark.catalog.tableExists(CONFIGURATIONS):
            spark.createDataFrame([], config_schema).write.format("delta").mode(
                "errorifexists"
            ).saveAsTable(CONFIGURATIONS)
        if (
            not spark.table(CONFIGURATIONS)
            .where(F.col("ConfigurationID") == configuration_id)
            .limit(1)
            .count()
        ):
            spark.createDataFrame(config_rows, config_schema).write.format(
                "delta"
            ).mode("append").saveAsTable(CONFIGURATIONS)
        print("Configuration snapshot:", configuration_id, "Rules:", len(rules))
        complete_sources = (
            spark.table(TABLE_PREFIX + "_file_state")
            .select("SourceRevisionID")
            .distinct()
        )
        source_measures = (
            spark.table(TABLE_PREFIX + "_source_measures")
            .join(complete_sources, "SourceRevisionID", "inner")
            .alias("h")
        )
        config = (
            spark.table(CONFIGURATIONS)
            .where(F.col("ConfigurationID") == configuration_id)
            .alias("c")
        )
        match = (
            (F.col("h.MeasureKey") == F.col("c.MeasureKey"))
            & (F.col("h.DemandCycle") >= F.col("c.EffectiveFromCycle"))
            & (F.col("h.DemandCycle") <= F.col("c.EffectiveToCycle"))
        )
        fields = [
            "DisplayName",
            "ReportingEnabled",
            "Family",
            "Dependency",
            "ValueRole",
            "AggregationRule",
            "ComparisonRule",
            "Unit",
            "EffectiveFromCycle",
            "EffectiveToCycle",
            "DefinitionStatus",
            "Owner",
            "ChangeReason",
        ]
        assignments = source_measures.join(config, match, "left").select(
            "h.*",
            *[F.col("c." + name) for name in fields],
            F.lit(configuration_id).alias("ConfigurationID"),
            F.when(F.col("c.MeasureKey").isNull(), "UNMAPPED")
            .otherwise("MAPPED")
            .alias("CatalogState"),
        )
        engine.replace_snapshot(
            TABLE_PREFIX + "_measure_assignments", assignments, ["SourceMeasureID"]
        )
        discovered = source_measures.select(
            "MeasureKey", "Section", "Measure"
        ).distinct()
        configured = spark.createDataFrame(
            [[r["MeasureKey"]] for r in rules],
            T.StructType([T.StructField("MeasureKey", T.StringType(), False)]),
        ).distinct()
        print(
            "Unmapped source measure identities:",
            discovered.join(configured, "MeasureKey", "left_anti").count(),
            flush=True,
        )
        engine.replace_snapshot(
            state_name,
            spark.createDataFrame(
                [(signature, datetime.now(UTC).isoformat())],
                "Signature string, CompletedAtUtc string",
            ),
            ["Signature"],
        )
        return True
    except Exception as error:
        print("Measure configuration was not captured:", str(error))
        print(
            "Raw bridge history remains available. Supply or correct the List/CSV connection before accepting configured reporting."
        )
        raise


In [ ]:
"""Reporting projection retains R03.1 arithmetic and adds folder-independent path aliases."""

from __future__ import annotations

import csv
import hashlib
import io
import json
from datetime import UTC, datetime
from typing import Any

from delta.tables import DeltaTable
from pyspark.sql import functions as F


def update_reporting(engine: Any, embedded_rules: str) -> bool:
    spark = engine.spark
    SOURCE_PREFIX = engine.prefix
    REPORT_PREFIX = engine.config.get("REPORT_PREFIX", "fm_review")
    RULES_PATH = engine.config.get(
        "REPORT_RULES_PATH", "Files/ForecastMasterControls/ReportMeasureRules.csv"
    )
    FORCE_REBUILD_REPORTING_FACTS = engine.config.get(
        "FORCE_REBUILD_REPORTING_FACTS", False
    )
    EMBEDDED_RULES_CSV = embedded_rules

    def utc_now() -> str:
        return datetime.now(UTC).isoformat()

    try:
        raw_rules = [
            r.asDict()
            for r in spark.read.option("header", True)
            .option("encoding", "UTF-8")
            .csv(RULES_PATH)
            .collect()
        ]
        print("Measure definitions:", RULES_PATH)
    except Exception as exc:
        if "PATH_NOT_FOUND" not in str(exc) and "does not exist" not in str(exc):
            raise
        raw_rules = list(csv.DictReader(io.StringIO(EMBEDDED_RULES_CSV)))
        print("Measure definitions: embedded initial seven-metric configuration")
    required_rules = {
        "MetricKey",
        "Section",
        "Measure",
        "Unit",
        "Aggregation",
        "CutoffFamily",
        "NumeratorKey",
        "NumeratorSection",
        "NumeratorMeasure",
        "NumeratorManufacturer",
        "NumeratorRequireProduct",
        "DenominatorKey",
        "DenominatorSection",
        "DenominatorMeasure",
        "DenominatorManufacturer",
        "DenominatorRequireProduct",
        "SortIndex",
    }
    raw_rules = [
        {k: "" if v is None else str(v) for k, v in r.items()} for r in raw_rules
    ]
    if not raw_rules or any(not required_rules <= set(r) for r in raw_rules):
        raise ValueError(
            "ReportMeasureRules.csv is missing required columns or contains no definitions"
        )
    if len({r["MetricKey"] for r in raw_rules}) != len(raw_rules):
        raise ValueError("MetricKey must be unique")
    components: dict[str, tuple[str, str, str, bool]] = {}
    for r in raw_rules:
        if not all(r[k] for k in ("MetricKey", "Section", "Measure", "NumeratorKey")):
            raise ValueError(
                "Each metric needs a nonempty identity, label and numerator"
            )
        r["SortIndex"] = str(int(r["SortIndex"]))
        if r["Aggregation"] not in ("SUM", "RATIO") or r["CutoffFamily"] not in (
            "IMS",
            "EF",
        ):
            raise ValueError("Unknown aggregation/cutoff definition: " + r["MetricKey"])
        if r["Unit"] not in ("Units", "£", "%", "Litres", "Pounds"):
            raise ValueError("Define an explicit supported Unit for " + r["MetricKey"])
        if r["Aggregation"] == "RATIO" and (not r["DenominatorKey"]):
            raise ValueError("Ratio requires a denominator: " + r["MetricKey"])
        for part in ("Numerator", "Denominator"):
            key = r[part + "Key"]
            if not key:
                continue
            if str(r[part + "RequireProduct"]).lower() not in ("true", "false"):
                raise ValueError("RequireProduct must be true or false for " + key)
            rule = (
                r[part + "Section"],
                r[part + "Measure"],
                r[part + "Manufacturer"],
                str(r[part + "RequireProduct"]).lower() == "true",
            )
            if not rule[0] or not rule[1]:
                raise ValueError(
                    "Exact original source section/measure required for " + key
                )
            if key in components and components[key] != rule:
                raise ValueError("Conflicting definitions for component " + key)
            components[key] = rule
    rule_hash = hashlib.sha256(
        json.dumps(raw_rules, sort_keys=True, ensure_ascii=False).encode()
    ).hexdigest()
    receipt_name = REPORT_PREFIX + "_projection_receipts"
    fact_name = REPORT_PREFIX + "_facts"
    source_rows = engine.report_sources()
    source_schema = ", ".join(k + " string" for k in source_rows[0])
    sources = spark.createDataFrame(source_rows, source_schema)
    report_signature = hashlib.sha256(
        json.dumps(
            {
                "rules": raw_rules,
                "sources": source_rows,
                "profiles": sorted(engine.allowed),
                "identityPolicy": "current-file-v1",
            },
            sort_keys=True,
        ).encode()
    ).hexdigest()
    state_name = REPORT_PREFIX + "_refresh_state"
    required_helpers = [
        REPORT_PREFIX + "_" + name
        for name in (
            "facts",
            "files",
            "reference",
            "measures",
            "months",
            "axis",
            "periods",
            "series",
            "modes",
        )
    ]
    if spark.catalog.tableExists(state_name) and all(
        spark.catalog.tableExists(name) for name in required_helpers
    ):
        signatures = {
            r[0] for r in spark.table(state_name).select("Signature").collect()
        }
        if signatures == {report_signature}:
            print(
                "Reporting sources and rules unchanged: no reporting tables rewritten.",
                flush=True,
            )
            return False
    if sources.limit(1).count() == 0:
        raise ValueError(
            "No completed source revisions in the configured source profile"
        )
    if sources.groupBy("SourceRevisionID").count().where("count <> 1").limit(1).count():
        raise ValueError("SourceRevisionID is not unique")
    existing_ids = []
    rebuild = FORCE_REBUILD_REPORTING_FACTS or not spark.catalog.tableExists(fact_name)
    if spark.catalog.tableExists(receipt_name) and (not rebuild):
        receipts = spark.table(receipt_name)
        hashes = {r[0] for r in receipts.select("RuleHash").distinct().collect()}
        if "ContentRevisionID" not in receipts.columns:
            rebuild = True
        if hashes != {rule_hash}:
            rebuild = True
            print(
                "Report definitions changed: rebuilding derived reporting facts from retained source history."
            )
        else:
            if not rebuild:
                completed = {
                    r["SourceRevisionID"]: r["ContentRevisionID"]
                    for r in receipts.collect()
                }
                existing_ids = [
                    r["SourceRevisionID"]
                    for r in source_rows
                    if completed.get(r["SourceRevisionID"]) == r["RawSourceRevisionID"]
                ]
    pending = (
        sources
        if rebuild or not existing_ids
        else sources.where(~F.col("SourceRevisionID").isin(existing_ids))
    )
    pending_ids = [r[0] for r in pending.select("SourceRevisionID").collect()]
    if pending_ids:
        component_schema = "ComponentKey string, SourceSection string, SourceMeasure string, ManufacturerRule string, RequireProduct boolean"
        cfg = spark.createDataFrame(
            [(k, *v) for k, v in components.items()], component_schema
        ).alias("c")
        pending_map = pending.select(
            "RawSourceRevisionID", F.col("SourceRevisionID").alias("ReportRevisionID")
        ).alias("s")
        raw_ids = [
            r[0] for r in pending_map.select("RawSourceRevisionID").distinct().collect()
        ]
        raw_history = (
            spark.table(SOURCE_PREFIX + "_history")
            .where(F.col("SourceRevisionID").isin(raw_ids))
            .alias("raw")
        )
        history = (
            raw_history.join(
                F.broadcast(pending_map),
                F.col("raw.SourceRevisionID") == F.col("s.RawSourceRevisionID"),
                "inner",
            )
            .select(
                *[
                    F.col("raw." + name)
                    for name in raw_history.columns
                    if name != "SourceRevisionID"
                ],
                F.col("s.ReportRevisionID").alias("SourceRevisionID"),
            )
            .alias("h")
        )
        matched = history.join(
            F.broadcast(cfg),
            (F.col("h.Section") == F.col("c.SourceSection"))
            & (F.col("h.Measure") == F.col("c.SourceMeasure")),
            "inner",
        )
        matched = matched.where(
            (F.col("c.ManufacturerRule") == "*")
            | (
                F.lower(F.trim(F.col("h.Manufacturer")))
                == F.lower(F.col("c.ManufacturerRule"))
            )
        )
        matched = matched.where(
            ~F.col("c.RequireProduct")
            | (F.length(F.trim(F.coalesce(F.col("h.Product"), F.lit("")))) > 0)
        )
        numeric = (F.col("ValueKind") == "number") & F.col("ValueNumber").isNotNull()
        projection = matched.groupBy(
            "SourceRevisionID",
            "Channel",
            "SubChannel",
            "Sector",
            F.col("DemandMonth").alias("Month"),
            "ComponentKey",
        ).agg(
            F.sum(F.when(numeric, F.col("ValueNumber"))).alias("Amount"),
            F.sum(F.when(numeric, 1).otherwise(0)).cast("long").alias("NumericCells"),
            F.sum(F.when(numeric, 0).otherwise(1)).cast("long").alias("MissingCells"),
        )
        projection = projection.persist()
        try:
            count = projection.count()
            if count == 0:
                raise ValueError(
                    "No fact components matched the exact source definitions; check source section/measure identities"
                )
            projected_ids = {
                r[0] for r in projection.select("SourceRevisionID").distinct().collect()
            }
            unmatched_ids = set(pending_ids) - projected_ids
            if unmatched_ids:
                raise ValueError(
                    "No configured fact components matched these source revisions: "
                    + ", ".join(sorted(unmatched_ids))
                )
            if rebuild:
                engine.replace_snapshot(
                    fact_name,
                    projection,
                    [
                        "SourceRevisionID",
                        "Channel",
                        "SubChannel",
                        "Sector",
                        "Month",
                        "ComponentKey",
                    ],
                )
            else:
                # Replace the complete affected-file slice atomically. MERGE insert-only
                # would retain removed months, dimensions, and previous numeric values.
                predicate = (
                    "SourceRevisionID IN ("
                    + ",".join("'" + sid + "'" for sid in pending_ids)
                    + ")"
                )
                detail = (
                    DeltaTable.forName(spark, fact_name)
                    .detail()
                    .select("location")
                    .first()
                )
                if detail is None:
                    raise ValueError("Reporting table location is unavailable")
                projection.coalesce(2).write.format("delta").mode("overwrite").option(
                    "replaceWhere", predicate
                ).save(detail[0])
                engine.writes[fact_name] += 1
            receipt_schema = "SourceRevisionID string, RuleHash string, ProjectedAtUtc string, ContentRevisionID string"
            stamp = datetime.now(UTC).isoformat()
            new_receipts = spark.createDataFrame(
                [
                    (r["SourceRevisionID"], rule_hash, stamp, r["RawSourceRevisionID"])
                    for r in source_rows
                    if r["SourceRevisionID"] in pending_ids
                ],
                receipt_schema,
            )
            if rebuild:
                # First R04.1 run migrates the legacy receipts and facts together.
                if (
                    spark.catalog.tableExists(receipt_name)
                    and "ContentRevisionID" not in spark.table(receipt_name).columns
                ):
                    spark.sql(
                        "ALTER TABLE "
                        + receipt_name
                        + " ADD COLUMNS (ContentRevisionID STRING)"
                    )
                engine.replace_snapshot(
                    receipt_name, new_receipts, ["SourceRevisionID"]
                )
            else:
                engine.upsert(
                    receipt_name, new_receipts, ["SourceRevisionID"], update=True
                )
            print(
                "Projected",
                len(pending_ids),
                "source revisions into",
                count,
                "monthly component rows.",
            )
        finally:
            projection.unpersist()
    else:
        print("All source revisions already projected. Raw history was not rescanned.")
    active_report_ids = [r["SourceRevisionID"] for r in source_rows]
    for name in (fact_name, receipt_name):
        stale = ~F.col("SourceRevisionID").isin(active_report_ids)
        if spark.table(name).where(stale).limit(1).count():
            DeltaTable.forName(spark, name).delete(stale)
            engine.writes[name] += 1
    files = sources.select(
        "SourceRevisionID",
        "SourceFileName",
        "SourceRelativePath",
        "Country",
        "DemandCycle",
        "GrowthDriver",
        "SourceProfile",
        F.to_date("IMSActualsThrough").alias("IMSActualsThrough"),
        F.to_date("ExFactoryActualsThrough").alias("ExFactoryActualsThrough"),
        "PublicationStatus",
    )
    files = files.join(
        sources.select("SourceRevisionID", "PathFirstSeenAtUtc"),
        "SourceRevisionID",
        "inner",
    )
    files = files.withColumn(
        "FileLabel",
        F.concat(
            F.col("SourceFileName"),
            F.lit(" · Imported "),
            F.col("PathFirstSeenAtUtc"),
            F.lit(" · "),
            F.col("SourceRelativePath"),
            F.lit(" · "),
            F.substring("SourceRevisionID", 1, 8),
        ),
    ).drop("PathFirstSeenAtUtc")
    if (
        files.where(
            F.col("IMSActualsThrough").isNull()
            | F.col("ExFactoryActualsThrough").isNull()
        )
        .limit(1)
        .count()
    ):
        raise ValueError("A source receipt has no usable IMS/Ex-factory cutoff date")

    def save_small(name: str, frame: Any) -> None:
        keys = {
            "files": "SourceRevisionID",
            "reference": "SourceRevisionID",
            "measures": "MetricKey",
            "months": "Month",
            "axis": "Month",
            "periods": "PeriodKey",
            "series": "SeriesID",
            "modes": "Mode",
        }
        engine.replace_snapshot(REPORT_PREFIX + "_" + name, frame, [keys[name]])

    save_small("reference", files)
    metric_fields = [
        "MetricKey",
        "Section",
        "Measure",
        "Unit",
        "Aggregation",
        "CutoffFamily",
        "NumeratorKey",
        "DenominatorKey",
        "SortIndex",
    ]
    metric_schema = ", ".join(
        k + (" long" if k == "SortIndex" else " string") for k in metric_fields
    )
    save_small(
        "measures",
        spark.createDataFrame(
            [
                [int(r[k]) if k == "SortIndex" else r[k] for k in metric_fields]
                for r in raw_rules
            ],
            metric_schema,
        ),
    )
    calendar = spark.table(SOURCE_PREFIX + "_calendar").where(
        F.col("SourceRevisionID").isin([r["RawSourceRevisionID"] for r in source_rows])
    )
    bounds = calendar.agg(
        F.min("DemandMonth").alias("first"), F.max("DemandMonth").alias("last")
    ).first()
    if not bounds or not bounds["first"] or (not bounds["last"]):
        raise ValueError("Source calendar is empty; date limits cannot be inferred")
    months = spark.range(1).select(
        F.explode(
            F.sequence(
                F.lit(bounds["first"]),
                F.lit(bounds["last"]),
                F.expr("interval 1 month"),
            )
        ).alias("Month")
    )
    months = months.select(
        "Month",
        F.date_format("Month", "MMM yyyy").alias("MonthLabel"),
        F.year("Month").cast("long").alias("Year"),
        F.month("Month").cast("long").alias("MonthNumber"),
        F.months_between("Month", F.lit("2000-01-01")).cast("long").alias("MonthIndex"),
    )
    save_small("months", months)
    save_small("axis", months)
    kinds = spark.createDataFrame(
        [("FY", 0), ("YTD", 1), ("YTG", 2)], "PeriodType string, KindOrder long"
    )
    periods = (
        months.select("Year")
        .distinct()
        .crossJoin(kinds)
        .select(
            "Year",
            "PeriodType",
            F.concat_ws(":", F.col("Year").cast("string"), "PeriodType").alias(
                "PeriodKey"
            ),
            F.concat_ws(" ", F.col("Year").cast("string"), "PeriodType").alias("Label"),
            (F.col("Year") * 3 + F.col("KindOrder")).alias("SortIndex"),
        )
    )
    save_small("periods", periods)
    base_series = spark.createDataFrame(
        [
            ("actual", "ACTUAL", "", "Actuals · reference", 0),
            ("trend12", "TREND12", "", "Trend 12M", 1),
            ("trend24", "TREND24", "", "Trend 24M", 2),
        ],
        "SeriesID string, SeriesKind string, SourceRevisionID string, SeriesLabel string, SortIndex long",
    )
    plans = (
        files.select(
            F.concat(F.lit("plan:"), "SourceRevisionID").alias("SeriesID"),
            F.lit("PLAN").alias("SeriesKind"),
            "SourceRevisionID",
            "FileLabel",
        )
        .withColumnRenamed("FileLabel", "SeriesLabel")
        .withColumn("SortIndex", F.lit(10).cast("long"))
    )
    save_small("series", base_series.unionByName(plans))
    save_small(
        "modes",
        spark.createDataFrame(
            [("Selected files", 0), ("Reference only", 1)],
            "Mode string, SortIndex long",
        ),
    )
    print(
        "Reporting helpers ready:",
        *[
            REPORT_PREFIX + "_" + s
            for s in (
                "facts",
                "files",
                "reference",
                "measures",
                "months",
                "axis",
                "periods",
                "series",
                "modes",
            )
        ],
        sep="\n",
    )
    print(
        "Reporting helpers updated. Existing report relationships and measures are reused."
    )
    save_small("files", files)
    engine.replace_snapshot(
        state_name,
        spark.createDataFrame(
            [(report_signature, utc_now())], "Signature string, CompletedAtUtc string"
        ),
        ["Signature"],
    )
    engine.report_changed = True
    print("Selectable report files:", len(source_rows), flush=True)
    return True


In [ ]:
"""Refresh the existing semantic model after successful reporting commits; never publish a model."""

from __future__ import annotations

import re
import time
from collections.abc import Callable
from typing import Any
from urllib.parse import urlparse

import requests


def refresh_model(
    config: dict[str, Any],
    get_token: Callable[[str], str],
    *,
    http: Any = requests,
    sleep: Callable[[float], None] = time.sleep,
    clock: Callable[[], float] = time.monotonic,
) -> dict[str, str]:
    if not config.get("REFRESH_SEMANTIC_MODEL", True):
        return {
            "Status": "MANUAL",
            "Message": "Refresh ForecastMaster Demand Model, then refresh Power BI Desktop.",
        }
    workspace, dataset = config["WORKSPACE_ID"], config["SEMANTIC_MODEL_ID"]
    if any(not re.fullmatch(r"[0-9a-fA-F-]{36}", x) for x in (workspace, dataset)):
        raise ValueError("Workspace and semantic-model IDs must be UUIDs")
    base = f"https://api.powerbi.com/v1.0/myorg/groups/{workspace}/datasets/{dataset}/refreshes"
    headers = {
        "Authorization": "Bearer " + get_token("pbi"),
        "Content-Type": "application/json",
    }
    response = http.post(
        base,
        headers=headers,
        json={"type": "automatic", "commitMode": "transactional", "retryCount": 1},
        timeout=45,
    )
    if response.status_code not in {200, 202}:
        raise RuntimeError(
            f"Semantic-model refresh not accepted (HTTP {response.status_code}): {response.text[:1200]}. Data tables are retained; refresh the existing model manually."
        )
    location = response.headers.get("Location", "")
    request_id = response.headers.get("x-ms-request-id", "")
    if not location and request_id:
        location = base + "/" + request_id
    parsed = urlparse(location)
    if (
        parsed.scheme != "https"
        or parsed.netloc != "api.powerbi.com"
        or not location.startswith(base + "/")
    ):
        return {
            "Status": "ACCEPTED_UNCONFIRMED",
            "Message": "Refresh was accepted; check model refresh history before refreshing the report.",
        }
    deadline = clock() + config.get("MODEL_REFRESH_TIMEOUT_SECONDS", 180)
    while clock() < deadline:
        state = http.get(location, headers=headers, timeout=45)
        if state.status_code not in {200, 202}:
            raise RuntimeError(
                f"Cannot confirm semantic-model refresh (HTTP {state.status_code}): {state.text[:1000]}"
            )
        status = state.json().get("status", "Unknown")
        if status == "Completed":
            return {"Status": "COMPLETED", "RequestID": request_id}
        if status in {"Failed", "Cancelled", "Disabled", "TimedOut"}:
            raise RuntimeError(
                "Semantic-model refresh ended with " + status + ": " + state.text[:1200]
            )
        sleep(2)
    return {
        "Status": "PENDING",
        "RequestID": request_id,
        "Message": "Accepted refresh is still running. Check model refresh history; rerunning ingestion is unnecessary.",
    }


In [ ]:
"""Single ordered invocation for the embedded Fabric notebook."""

import json
from typing import Any



def run_refresh(
    spark: Any,
    reader_namespace: dict[str, Any],
    config: dict[str, Any],
    notebookutils: Any,
    embedded_rules: str,
) -> tuple[FastRefresh, dict[str, Any]]:
    engine = FastRefresh(spark, reader_namespace, config)
    old_shuffle = spark.conf.get("spark.sql.shuffle.partitions")
    spark.conf.set(
        "spark.sql.shuffle.partitions", str(config.get("SHUFFLE_PARTITIONS", 8))
    )
    model_result = {"Status": "NOT_REQUESTED"}
    try:
        try:
            engine.ingest()
        except Exception as exc:  # noqa: BLE001 -- retain stage diagnostics and surface failure
            engine.stage_errors.append("Ingestion: " + str(exc))
            print("Ingestion stage failed:", str(exc), flush=True)
        if engine.completed:
            try:
                with engine.stage("Update measure catalog"):
                    update_catalog(engine)
            except Exception as exc:  # noqa: BLE001 -- retain stage diagnostics and surface failure
                engine.stage_errors.append("Catalog: " + str(exc))
            try:
                with engine.stage("Update changed reporting projections"):
                    update_reporting(engine, embedded_rules)
                if engine.report_changed or config.get("FORCE_MODEL_REFRESH", False):
                    with engine.stage("Refresh ForecastMaster Demand Model"):
                        model_result = refresh_model(
                            config, notebookutils.credentials.getToken
                        )
                    if model_result["Status"] != "COMPLETED":
                        engine.stage_errors.append(
                            "Model refresh: " + json.dumps(model_result)
                        )
                else:
                    model_result = {
                        "Status": "UNCHANGED",
                        "Message": "Report tables unchanged; no model refresh needed.",
                    }
            except Exception as exc:  # noqa: BLE001 -- retain stage diagnostics and surface failure
                engine.stage_errors.append("Reporting/model refresh: " + str(exc))
                print("Reporting/model stage:", str(exc), flush=True)
        if (
            model_result["Status"] in {"COMPLETED", "UNCHANGED"}
            and not engine.stage_errors
        ):
            try:
                with engine.stage("Compact superseded file data"):
                    engine.compact_current_sources()
            except Exception as exc:  # noqa: BLE001 -- report retention failure honestly
                engine.stage_errors.append("Current-file compaction: " + str(exc))
        summary = engine.summary()
        summary["ModelRefresh"] = model_result
        print(json.dumps(summary, indent=2), flush=True)
        if summary["Status"] == "SUCCESS":
            print(
                "Refresh complete. In Power BI Desktop, refresh the report to retrieve the updated model results.",
                flush=True,
            )
        else:
            print(
                "PARTIAL: inspect FAILED file rows and StageErrors. Successful existing sources remain available.",
                flush=True,
            )
        return engine, summary
    finally:
        spark.conf.set("spark.sql.shuffle.partitions", old_shuffle)


In [ ]:
"""Confirm a model refresh using its operation URL or fresh API history.

History fallback assumes this worker is the sole API refresh writer. It never
accepts a baseline row, an earlier start time, or multiple fresh API requests.
"""

from __future__ import annotations

import time
from collections.abc import Callable
from datetime import UTC, datetime
from email.utils import parsedate_to_datetime
from typing import Any
from urllib.parse import urlparse
from uuid import UUID

import requests


def model_history_key(row: dict[str, Any]) -> str:
    return str(row.get("requestId") or row.get("id") or "")


def model_start_time(value: Any) -> datetime | None:
    try:
        result = datetime.fromisoformat(str(value))
        return result.astimezone(UTC) if result.tzinfo is not None else None
    except (ValueError, TypeError):
        return None


def refresh_confirmed_model(
    config: dict[str, Any],
    get_token: Callable[[str], str],
    *,
    http: Any = requests,
    sleep: Callable[[float], None] = time.sleep,
    clock: Callable[[], float] = time.monotonic,
    now: Callable[[], datetime] = lambda: datetime.now(UTC),
) -> dict[str, str]:
    if not config.get("REFRESH_SEMANTIC_MODEL", True):
        return {"Status": "MANUAL", "Message": "Semantic-model refresh is disabled."}
    workspace = str(UUID(config["WORKSPACE_ID"]))
    dataset = str(UUID(config["SEMANTIC_MODEL_ID"]))
    base = f"https://api.powerbi.com/v1.0/myorg/groups/{workspace}/datasets/{dataset}/refreshes"

    def headers() -> dict[str, str]:
        return {"Authorization": "Bearer " + get_token("pbi")}

    def history() -> tuple[list[dict[str, Any]], Any]:
        result = http.get(
            base,
            params={"$top": 60},
            headers=headers(),
            timeout=45,
            allow_redirects=False,
        )
        if result.status_code != 200:
            raise RuntimeError(
                f"Cannot read model refresh history (HTTP {result.status_code})."
            )
        rows = result.json().get("value")
        if not isinstance(rows, list) or any(not isinstance(r, dict) for r in rows):
            raise ValueError("Invalid model refresh history response.")
        return rows, result

    baseline, baseline_response = history()
    known = {model_history_key(row) for row in baseline}
    started = now().astimezone(UTC)
    server_date = baseline_response.headers.get("Date")
    if server_date:
        started = parsedate_to_datetime(server_date).astimezone(UTC)
    submitted = http.post(
        base,
        headers={**headers(), "Content-Type": "application/json"},
        json={"type": "automatic", "commitMode": "transactional", "retryCount": 1},
        timeout=45,
        allow_redirects=False,
    )
    if submitted.status_code not in {200, 202}:
        raise RuntimeError(
            f"Semantic-model refresh was rejected (HTTP {submitted.status_code}): {submitted.text[:1000]}"
        )
    # A service timestamp avoids depending on the notebook clock during fallback.
    if submitted.headers.get("Date"):
        started = parsedate_to_datetime(submitted.headers["Date"]).astimezone(UTC)
    location = submitted.headers.get("Location", "")
    operation = ""
    parsed = urlparse(location)
    if (
        parsed.scheme == "https"
        and parsed.netloc == "api.powerbi.com"
        and location.startswith(base + "/")
        and not parsed.query
        and not parsed.fragment
    ):
        tail = location[len(base) + 1 :]
        try:
            operation = str(UUID(tail))
        except ValueError:
            location = ""
    else:
        location = ""
    header_id = submitted.headers.get("x-ms-request-id", "")
    deadline = clock() + float(config.get("MODEL_REFRESH_TIMEOUT_SECONDS", 180))
    previous_candidate = ""
    terminal = {"Failed", "Cancelled", "Canceled", "Disabled", "TimedOut"}
    while clock() < deadline:
        if location:
            detail = http.get(
                location, headers=headers(), timeout=45, allow_redirects=False
            )
            if detail.status_code == 200:
                body = detail.json()
                identity = str(body.get("requestId") or operation)
                if identity != operation:
                    raise ValueError("Model refresh operation identity changed.")
                status = body.get("status", "Unknown")
                if status == "Completed":
                    return {
                        "Status": "COMPLETED",
                        "RequestID": operation,
                        "Confirmation": "OperationURL",
                    }
                if status in terminal:
                    raise RuntimeError(
                        "Semantic-model refresh ended with " + str(status)
                    )
            elif detail.status_code not in {202, 400, 404, 429, 500, 502, 503, 504}:
                raise RuntimeError(
                    f"Cannot confirm model refresh (HTTP {detail.status_code})."
                )
        rows, _ = history()
        correlated = [
            r
            for r in rows
            if model_history_key(r) not in known
            and model_history_key(r) in {operation, header_id} - {""}
        ]
        method = "RequestID"
        if not correlated:
            correlated = [
                r
                for r in rows
                if model_history_key(r)
                and model_history_key(r) not in known
                and r.get("refreshType") in {"ViaApi", "ViaEnhancedApi"}
                and (
                    model_start_time(r.get("startTime"))
                    or datetime.min.replace(tzinfo=UTC)
                )
                >= started
            ]
            method = "FreshAPIHistorySoleWriter"
        if len(correlated) > 1:
            return {
                "Status": "ACCEPTED_UNCONFIRMED",
                "Message": "Multiple fresh API refreshes found. Inspect model history; no request was marked successful.",
            }
        if len(correlated) == 1:
            row = correlated[0]
            identity = model_history_key(row)
            status = row.get("status", "Unknown")
            if status in terminal:
                raise RuntimeError("Semantic-model refresh ended with " + str(status))
            # Require a second observation for fallback correlation.
            if status == "Completed" and (
                method == "RequestID" or identity == previous_candidate
            ):
                return {
                    "Status": "COMPLETED",
                    "RequestID": identity,
                    "Confirmation": method,
                }
            previous_candidate = identity
        sleep(2)
    return {
        "Status": "PENDING",
        "RequestID": operation or header_id,
        "Message": "Model completion was not confirmed within the wait limit. Inspect refresh history.",
    }


In [ ]:
EMBEDDED_REPORT_RULES = 'MetricKey,Section,Measure,Unit,Aggregation,CutoffFamily,NumeratorKey,NumeratorSection,NumeratorMeasure,NumeratorManufacturer,NumeratorRequireProduct,DenominatorKey,DenominatorSection,DenominatorMeasure,DenominatorManufacturer,DenominatorRequireProduct,SortIndex\nims-market-units,IMS / Market,IMS Market Units,Units,SUM,IMS,product-units,Pricing,Product Selling units,*,true,,,,,false,10\nims-abbott-units,IMS / Market,IMS Abbott Units,Units,SUM,IMS,abbott-units,Pricing,Product Selling units,Abbott,true,,,,,false,20\nims-volume-share,IMS / Market,IMS Abbott Volume Share,%,RATIO,IMS,abbott-units,Pricing,Product Selling units,Abbott,true,product-units,Pricing,Product Selling units,*,true,30\nims-market-value,IMS / Market,IMS Market Value,£,SUM,IMS,product-value,Pricing,Product Value £,*,true,,,,,false,40\nims-value-share,IMS / Market,IMS Abbott Value Share,%,RATIO,IMS,abbott-value,Pricing,Product Value £,Abbott,true,product-value,Pricing,Product Value £,*,true,50\nef-selling-units,Ex-Factory,Ex-factory Selling Units,Units,SUM,EF,ef-ads-units,Ex-Factory,ADS Selling units,*,false,,,,,false,60\nef-value,Ex-Factory,Ex-factory Value,£,SUM,EF,ef-ads-value,Ex-Factory,ADS Value £,*,false,,,,,false,70\n'
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

# The original ordered ingestion/reporting implementation calls this binding.
refresh_model = refresh_confirmed_model

config_keys = ["SOURCE_ROOT","TABLE_PREFIX","REPORT_PREFIX","ALLOWED_PROFILES","REFRESH_MODE","SELECTED_RELATIVE_PATHS","FORCE_REHASH","FORCE_REBUILD_REPORTING_FACTS","FORCE_MODEL_REFRESH","REFRESH_SEMANTIC_MODEL","WORKSPACE_ID","SEMANTIC_MODEL_ID","MAX_BRIDGE_BYTES","MAX_OBSERVATIONS_PER_FILE","SHUFFLE_PARTITIONS","MODEL_REFRESH_TIMEOUT_SECONDS"]
def requested_refresh(force_paths=None):
    refresh_config = {key: globals()[key] for key in config_keys}
    # Requests always cover the complete naming-matched Demand tree.
    refresh_config.update(
        SOURCE_ROOT="Files/Demand", REFRESH_MODE="ALL", SELECTED_RELATIVE_PATHS=[],
        ALLOWED_PROFILES=["Local", "Ext"], REFRESH_SEMANTIC_MODEL=True,
        FORCE_MODEL_REFRESH=(fm_mode == "REQUESTS"), FORCE_REHASH_PATHS=list(force_paths or []),
    )
    return run_refresh(spark, globals(), refresh_config, notebookutils, EMBEDDED_REPORT_RULES)

if fm_mode == "HOURLY":
    engine, worker_result = hourly_worker.run(fm_hourly_snapshot, requested_refresh)
else:
    engine, worker_result = request_worker.run(requested_refresh)
if engine is not None:
    if engine.events:
        display(engine.string_frame([row for row in engine.events if row["Status"] != "OUTSIDE_FILENAME_PATTERN"], EVENT_FIELDS))
    if engine.timings:
        display(spark.createDataFrame(engine.timings))
print("WORKER RESULT:", json.dumps(worker_result, indent=2), flush=True)

notebookutils.notebook.exit(json.dumps(worker_result, ensure_ascii=False, allow_nan=False))
